# BERT

In [ ]:
!curl -L -o ~/invoices-ocr.zip https://www.kaggle.com/api/v1/datasets/download/osamahosamabdellatif/high-quality-invoice-images-for-ocr
!unzip ~/invoices-ocr.zip -d /content/invoices

In [ ]:
!pip install -q \
    transformers \
    datasets \
    accelerate \
    sentencepiece \
    scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [33]:
!pip install -q --use-pep517 seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
import json
import random
import re

import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
import unicodedata

from pydantic_core import core_schema

from datasets import Dataset, DatasetDict
from collections.abc import Callable
from collections import defaultdict
from sklearn.model_selection import train_test_split
from seqeval.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from pydantic import BaseModel
from enum import StrEnum

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [39]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA: True
GPU: NVIDIA A100-SXM4-40GB


In [5]:
DATA_PATH = Path("/content/drive/MyDrive/ocr/annotated_invoices.parquet")

In [7]:
columns = [
    "path",
    "file_name",
    "subdir",
    "source_group",
    "ocred_text",
    "json_data",
]

In [6]:
df = pd.read_parquet(DATA_PATH)
df.head()

,file_name,json_data,ocred_text,subdir,source_group,csv_path,path,target,top_level_keys,target_text
0,batch1-0494.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Clark...",Invoice no: 84652373 Date of issue: 02/23/2021...,batch1_1,batch_1/batch_1,/content/invoices/batch_1/batch_1/batch1_1.csv,/content/invoices/batch_1/batch_1/batch1_1/bat...,{'invoice': {'client_address': '77477 Troy Cli...,"[invoice, items, payment_instructions, subtotal]","{""invoice"":{""client_address"":""77477 Troy Cliff..."
1,batch1-0489.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Willi...",Invoice no: 37451664 Date of issue: 06/11/2020...,batch1_1,batch_1/batch_1,/content/invoices/batch_1/batch_1/batch1_1.csv,/content/invoices/batch_1/batch_1/batch1_1/bat...,{'invoice': {'client_address': '588 Morales Ra...,"[invoice, items, payment_instructions, subtotal]","{""invoice"":{""client_address"":""588 Morales Rapi..."
2,batch1-0499.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Heste...",Invoice no: 40108666 Date of issue: 02/07/2020...,batch1_1,batch_1/batch_1,/content/invoices/batch_1/batch_1/batch1_1.csv,/content/invoices/batch_1/batch_1/batch1_1/bat...,{'invoice': {'client_address': '4000 Tonya Rap...,"[invoice, items, payment_instructions, subtotal]","{""invoice"":{""client_address"":""4000 Tonya Rapid..."
3,batch1-0497.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Olson...",Invoice no: 73285932 Date of issue: 07/25/2017...,batch1_1,batch_1/batch_1,/content/invoices/batch_1/batch_1/batch1_1.csv,/content/invoices/batch_1/batch_1/batch1_1/bat...,{'invoice': {'client_address': '80943 Beck Div...,"[invoice, items, payment_instructions, subtotal]","{""invoice"":{""client_address"":""80943 Beck Divid..."
4,batch1-0081.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Wilso...",Invoice no: 15288019 Date of issue: 09/07/2014...,batch1_1,batch_1/batch_1,/content/invoices/batch_1/batch_1/batch1_1.csv,/content/invoices/batch_1/batch_1/batch1_1/bat...,{'invoice': {'client_address': '85455 Carter E...,"[invoice, items, payment_instructions, subtotal]","{""invoice"":{""client_address"":""85455 Carter Est..."


In [193]:
PHASE_1_FIELDS = [
    "invoice.invoice_number",
    "invoice.invoice_date",
    "invoice.due_date",
    "invoice.seller_name",
    "invoice.seller_address",
    "invoice.client_name",
    "invoice.client_address",
    "subtotal.tax",
    "subtotal.discount",
    "subtotal.total",
    "payment_instructions.bank_name",
    "payment_instructions.account_number",
    "payment_instructions.payment_method",
]

ENTITY_TYPES = [
    "INVOICE_NUMBER",
    "INVOICE_DATE",
    "INVOICE_DUE_DATE",
    "SELLER_NAME",
    "SELLER_ADDRESS",
    "CLIENT_NAME",
    "CLIENT_ADDRESS",
    "TAX",
    "DISCOUNT",
    "TOTAL",
    "PAYMENT_DUE_DATE",
    "BANK_NAME",
    "ACCOUNT_NUMBER",
    "PAYMENT_METHOD",
]

BASELINE_ENTITIES = [
    "INVOICE_NUMBER",
    "INVOICE_DATE",
    "SELLER_NAME",
    "SELLER_ADDRESS",
    "CLIENT_NAME",
    "CLIENT_ADDRESS",
    "TAX",
    "TOTAL",
]

ENTITY_TYPES = BASELINE_ENTITIES

LABELS = ["O"]

for entity in ENTITY_TYPES:
    LABELS.extend([
        f"B-{entity}",
        f"I-{entity}",
    ])

label2id = {
    label: index
    for index, label in enumerate(LABELS)
}

id2label = {
    index: label
    for label, index in label2id.items()
}

print(LABELS)

['O', 'B-INVOICE_NUMBER', 'I-INVOICE_NUMBER', 'B-INVOICE_DATE', 'I-INVOICE_DATE', 'B-SELLER_NAME', 'I-SELLER_NAME', 'B-SELLER_ADDRESS', 'I-SELLER_ADDRESS', 'B-CLIENT_NAME', 'I-CLIENT_NAME', 'B-CLIENT_ADDRESS', 'I-CLIENT_ADDRESS', 'B-TAX', 'I-TAX', 'B-TOTAL', 'I-TOTAL']


## Invoice model

In [ ]:
@dataclass(frozen=True, slots=True)
class CandidateSpan:
    start: int
    end: int

    def overlaps(
        self,
        other: "CandidateSpan",
    ) -> bool:
        return (
            self.start < other.end
            and self.end > other.start
        )

In [ ]:
def normalize_with_mapping(text):
    normalized = []
    original_positions = []

    previous_was_space = True

    for index, character in enumerate(str(text)):
        if character.isalnum():
            normalized.append(character.lower())
            original_positions.append(index)
            previous_was_space = False

        elif not previous_was_space:
            normalized.append(" ")
            original_positions.append(index)
            previous_was_space = True

    if normalized and normalized[-1] == " ":
        normalized.pop()
        original_positions.pop()

    return "".join(normalized), original_positions

In [ ]:
class Field(str):
    @classmethod
    def __get_pydantic_core_schema__(
        cls,
        source_type,
        handler,
    ):
        return (
            core_schema
            .no_info_after_validator_function(
                cls,
                core_schema.str_schema(),
            )
        )

    def normalized(self) -> str:
        value = unicodedata.normalize(
            "NFKC",
            self,
        )

        return re.sub(
            r"\s+",
            " ",
            value,
        ).strip()
    
    def alignment_parts(
        self,
    ) -> tuple["Field", ...]:
        return (self,)

    def find_matches(
        self,
        text: str,
    ) -> list[CandidateSpan]:
        normalized_text, positions = (
            normalize_with_mapping(text)
        )
        normalized_value, _ = (
            normalize_with_mapping(self)
        )

        if not normalized_value:
            return []

        matches = []
        search_start = 0

        while True:
            normalized_start = normalized_text.find(
                normalized_value,
                search_start,
            )

            if normalized_start == -1:
                break

            normalized_end = (
                normalized_start
                + len(normalized_value)
            )

            matches.append(
                CandidateSpan(
                    start=positions[normalized_start],
                    end=(
                        positions[normalized_end - 1]
                        + 1
                    ),
                )
            )

            search_start = normalized_start + 1

        return matches


class MoneyField(Field):
    def digits(self) -> str:
        return "".join(
            re.findall(r"\d", self)
        )

    def find_matches(
        self,
        text: str,
    ) -> list[CandidateSpan]:
        digits = self.digits()

        if not digits:
            return []

        pattern = (
            r"(?<!\d)"
            + r"[\s,.'’]*".join(
                re.escape(digit)
                for digit in digits
            )
            + r"(?!\d)"
        )

        return [
            CandidateSpan(
                start=match.start(),
                end=match.end(),
            )
            for match in re.finditer(
                pattern,
                text,
            )
        ]

class IdentifierField(Field):
    def compact(self) -> str:
        return re.sub(
            r"[^A-Za-z0-9]",
            "",
            self,
        ).upper()

    def find_matches(
        self,
        text: str,
    ) -> list[CandidateSpan]:
        characters = re.findall(
            r"[A-Za-z0-9]",
            self,
        )

        if not characters:
            return []

        pattern = (
            r"(?<![A-Za-z0-9])"
            + r"[^A-Za-z0-9]*".join(
                re.escape(character)
                for character in characters
            )
            + r"(?![A-Za-z0-9])"
        )

        return [
            CandidateSpan(
                start=match.start(),
                end=match.end(),
            )
            for match in re.finditer(
                pattern,
                text,
                flags=re.IGNORECASE,
            )
        ]

class AddressField(Field):
    def alignment_parts(self) -> tuple["AddressField", ...]:
        return tuple(
            type(self)(line.strip())
            for line in self.splitlines()
            if line.strip()
        )

In [ ]:
class EntityType(StrEnum):
    INVOICE_NUMBER = "INVOICE_NUMBER"
    INVOICE_DATE = "INVOICE_DATE"
    INVOICE_DUE_DATE = "INVOICE_DUE_DATE"

    SELLER_NAME = "SELLER_NAME"
    SELLER_ADDRESS = "SELLER_ADDRESS"

    CLIENT_NAME = "CLIENT_NAME"
    CLIENT_ADDRESS = "CLIENT_ADDRESS"

    TAX = "TAX"
    TOTAL = "TOTAL"
    DISCOUNT = "DISCOUNT"

    PAYMENT_DUE_DATE = "PAYMENT_DUE_DATE"
    ACCOUNT_NUMBER = "ACCOUNT_NUMBER"
    BANK_NAME = "BANK_NAME"
    PAYMENT_METHOD = "PAYMENT_METHOD"
    


class InvoiceData(BaseModel):
    invoice_number: IdentifierField | None = None
    invoice_date: Field | None = None
    invoice_due_date: Field | None = None
    seller_name: Field | None = None
    seller_address: AddressField | None = None
    client_name: Field | None = None
    client_address: AddressField | None = None


class SubtotalData(BaseModel):
    tax: MoneyField | None = None
    total: MoneyField | None = None
    discount: MoneyField | None = None


class PaymentInstructions(BaseModel):
    due_date: Field | None = None
    account_number: IdentifierField | None = None
    bank_name: Field | None = None
    payment_method: Field | None = None


class InvoiceTarget(BaseModel):
    invoice: InvoiceData
    subtotal: SubtotalData
    payment_instructions: PaymentInstructions
    items: list[dict]

    def extract_fields(self) -> list[tuple[EntityType, Field]]:
        fields = [
            ( EntityType.INVOICE_NUMBER, self.invoice.invoice_number ),
            ( EntityType.INVOICE_DATE, self.invoice.invoice_date ),
            ( EntityType.INVOICE_DUE_DATE, self.invoice.invoice_due_date ),
            ( EntityType.SELLER_NAME, self.invoice.seller_name ),
            ( EntityType.SELLER_ADDRESS, self.invoice.seller_address ),
            ( EntityType.CLIENT_NAME, self.invoice.client_name ),
            ( EntityType.CLIENT_ADDRESS, self.invoice.client_address ),
            ( EntityType.TAX, self.subtotal.tax ),
            ( EntityType.DISCOUNT, self.subtotal.discount ),
            ( EntityType.TOTAL, self.subtotal.total ),
            ( EntityType.PAYMENT_DUE_DATE, self.payment_instructions.due_date ),
            ( EntityType.BANK_NAME, self.payment_instructions.bank_name ),
            ( EntityType.ACCOUNT_NUMBER, self.payment_instructions.account_number ),
            ( EntityType.PAYMENT_METHOD, self.payment_instructions.payment_method )
        ]
        return [
            (entity, value)
            for entity, value in fields
            if value is not None
        ]

@dataclass(frozen=True, slots=True)
class EntitySpan:
    """
    Represents entity span for each particular entity
    """

    start: int
    end: int
    entity: str
    value: str

    def __post_init__(self):
        if self.start < 0:
            raise ValueError(
                "Span start cannot be negative"
            )

        if self.end <= self.start:
            raise ValueError(
                f"Invalid span: [{self.start}, {self.end})"
            )

        if not self.entity:
            raise ValueError(
                "Entity type cannot be empty"
            )

    @property
    def length(self) -> int:
        return self.end - self.start

    def overlaps(
        self,
        other: "EntitySpan",
    ) -> bool:
        return (
            self.start < other.end
            and self.end > other.start
        )

    def overlaps_range(
        self,
        start: int,
        end: int,
    ) -> bool:
        return (
            self.start < end
            and self.end > start
        )

In [ ]:
@dataclass(frozen=True, slots=True)
class FinancialSummarySpans:
    net: CandidateSpan
    tax: CandidateSpan
    total: CandidateSpan

    def get(
        self,
        entity: EntityType,
    ) -> CandidateSpan | None:
        match entity:
            case EntityType.TAX:
                return self.tax

            case EntityType.TOTAL:
                return self.total

            case _:
                return None


@dataclass
class CandidatesSelector:
    def _select_unique_candidate(
        self,
        text: str,
        candidates: list[CandidateSpan],
    ) -> CandidateSpan | None:
        if len(candidates) != 1:
            return None

        return candidates[0]

    def _select_final_total_candidate(
        self,
        text: str,
        candidates: list[CandidateSpan],
    ) -> CandidateSpan | None:
        if not candidates:
            return None

        anchors = list(
            re.finditer(
                r"\btotal\b",
                text,
                flags=re.IGNORECASE,
            )
        )

        if anchors:
            final_anchor = anchors[-1].start()

            candidates_after_total = [
                candidate
                for candidate in candidates
                if candidate.start > final_anchor
            ]

            if candidates_after_total:
                return candidates_after_total[-1]

        return candidates[-1]

    def select(
        self,
        text: str,
        entity_type: EntityType,
        candidates: list[CandidateSpan]
    ) -> CandidateSpan | None:
        if entity_type in {EntityType.TAX, EntityType.TOTAL}:
            return self._select_final_total_candidate(
                text=text,
                candidates=candidates
            )
        return self._select_unique_candidate(
            text=text,
            candidates=candidates
        )

CandidateSelector = Callable[
    [
        str,
        list[CandidateSpan],
    ],
    CandidateSpan | None,
]

def select_unique_candidate(
    text: str,
    candidates: list[CandidateSpan],
) -> CandidateSpan | None:
    if len(candidates) != 1:
        return None

    return candidates[0]

def select_final_total_candidate(
    text: str,
    candidates: list[CandidateSpan],
) -> CandidateSpan | None:
    if not candidates:
        return None

    anchors = list(
        re.finditer(
            r"\btotal\b",
            text,
            flags=re.IGNORECASE,
        )
    )

    if anchors:
        final_anchor = anchors[-1].start()

        candidates_after_total = [
            candidate
            for candidate in candidates
            if candidate.start > final_anchor
        ]

        if candidates_after_total:
            return candidates_after_total[-1]

    return candidates[-1]

ENTITY_SELECTORS: dict[
    EntityType,
    CandidateSelector,
] = {
    EntityType.TAX: select_final_total_candidate,
    EntityType.TOTAL: select_final_total_candidate
}

class FinancialSummaryAligner:
    MONEY_PATTERN = re.compile(
        r"""
        (?<![\dA-Za-z])
        (
            (?:
                \d{1,3}
                (?:[ \u00A0.'’]\d{3})+
                |
                \d+
            )
            [,.]
            \d{2}
        )
        (?!\d)
        """,
        flags=re.VERBOSE,
    )

    TOTAL_PATTERN = re.compile(
        r"\btotal\b",
        flags=re.IGNORECASE,
    )

    def __init__(
        self,
        lookahead: int = 200,
    ):
        self.lookahead = lookahead

    def find(
        self,
        text: str,
    ) -> FinancialSummarySpans | None:
        total_anchors = list(
            self.TOTAL_PATTERN.finditer(text)
        )

        # Start from the final occurrence because "total" can
        # appear in product descriptions or other sections.
        for anchor in reversed(total_anchors):
            region_end = min(
                len(text),
                anchor.end() + self.lookahead,
            )

            amounts = list(
                self.MONEY_PATTERN.finditer(
                    text,
                    anchor.end(),
                    region_end,
                )
            )

            if len(amounts) < 3:
                continue

            net_match = amounts[0]
            tax_match = amounts[1]
            total_match = amounts[2]

            return FinancialSummarySpans(
                net=CandidateSpan(
                    start=net_match.start(),
                    end=net_match.end(),
                ),
                tax=CandidateSpan(
                    start=tax_match.start(),
                    end=tax_match.end(),
                ),
                total=CandidateSpan(
                    start=total_match.start(),
                    end=total_match.end(),
                ),
            )

        return None

@dataclass
class DocumentAligner:
    financial_summary_aligner: FinancialSummaryAligner
    selector: CandidatesSelector

    def find_spans(self, text: str, target: InvoiceTarget) -> list[EntitySpan]:
        
        financial_summary = (
            self.financial_summary_aligner.find(text)
        )
        spans: list[EntitySpan] = []
        occupied: list[CandidateSpan] = []

        for entity, value in target.extract_fields():

            for value_part in value.alignment_parts():
                financial_candidate = (
                    financial_summary.get(entity)
                    if financial_summary is not None
                    else None
                )
                if financial_candidate is not None:
                    # For TAX and TOTAL, the position in the final
                    # summary row is more reliable than ordinary
                    # text matching.
                    candidates = [financial_candidate]
                else:
                    candidates = value_part.find_matches(text)

                available_candidates = [
                    candidate
                    for candidate in candidates
                    if not any(
                        candidate.overlaps(existing)
                        for existing in occupied
                    )
                ]

                if not available_candidates:
                    continue

                if financial_candidate is not None:
                    # The financial aligner already selected the
                    # semantic position.
                    selected = available_candidates[0]
                else:
                    selected = self.selector.select(
                        text=text,
                        entity_type=entity,
                        candidates=available_candidates
                    )

                if selected is None:
                    continue

                spans.append(
                    EntitySpan(
                        start=selected.start,
                        end=selected.end,
                        entity=entity,
                        # Preserve the OCR-observed text rather than
                        # copying the expected JSON value.
                        value=text[
                            selected.start:selected.end
                        ],
                    )
                )

                occupied.append(selected)

        return sorted(
            spans,
            key=lambda span: span.start,
        )

### Testing

In [ ]:
row = df.iloc[1]
target = InvoiceTarget.model_validate(
    json.loads(row["json_data"])
)

target.invoice.seller_name
target.subtotal.tax
target.payment_instructions.account_number

## Reporting

In [ ]:
from collections.abc import Iterable
from collections.abc import Hashable

def group_spans_by_entity(
    spans: Iterable[EntitySpan],
) -> dict[
    EntityType,
    tuple[EntitySpan, ...],
]:
    grouped: dict[
        EntityType,
        list[EntitySpan],
    ] = defaultdict(list)

    for span in spans:
        grouped[span.entity].append(span)

    return {
        entity: tuple(
            sorted(
                entity_spans,
                key=lambda span: span.start,
            )
        )
        for entity, entity_spans
        in grouped.items()
    }

def money_digits(value: object) -> str:
    return "".join(
        character
        for character in str(value)
        if character.isdigit()
    )

@dataclass
class DocumentReportRow:
    row_index: Hashable
    file_name: str
    expected_entities: int
    matched_entities: int
    alignment_coverage: float
    missing_entities: tuple[str, ...]

    @classmethod
    def create(
        cls,
        *,
        row_index: Hashable,
        file_name: str,
        expected_fields: tuple[
            tuple[EntityType, Field],
            ...
        ],
        spans: tuple[EntitySpan, ...],
    ) -> "DocumentReportRow":
        expected = {
            entity
            for entity, _ in expected_fields
        }

        matched = {
            span.entity
            for span in spans
        }

        matched_expected = expected & matched

        return cls(
            row_index=row_index,
            file_name=file_name,
            expected_entities=len(expected),
            matched_entities=len(
                matched_expected
            ),
            alignment_coverage=(
                len(matched_expected) / len(expected)
                if expected
                else 1.0
            ),
            missing_entities=tuple(
                entity.value
                for entity in sorted(
                    expected - matched,
                    key=lambda item: item.value,
                )
            ),
        )


@dataclass
class DetailReportRow:
    row_index: Hashable
    file_name: str
    entity: str
    expected: str
    observed: str | None
    aligned: bool
    span_count: int
    parts: int
    matched_parts: int
    fully_raw_matched: bool
    partially_raw_matched: bool
    ambiguous: bool
    candidate_counts: tuple[int, ...]
    selected_spans: tuple[EntitySpan, ...]

    @classmethod
    def create(
        cls,
        *,
        row_index: Hashable,
        file_name: str,
        text: str,
        entity: EntityType,
        value: Field,
        selected_spans: tuple[
            EntitySpan,
            ...
        ],
    ) -> "DetailReportRow":
        value_parts = value.alignment_parts()

        part_candidates = [
            part.find_matches(text)
            for part in value_parts
        ]

        matched_parts = sum(
            bool(candidates)
            for candidates in part_candidates
        )

        candidate_counts = tuple(
            len(candidates)
            for candidates in part_candidates
        )

        return cls(
            row_index=row_index,
            file_name=file_name,
            entity=entity.value,
            expected=str(value),
            observed=(
                "\n".join(
                    span.value
                    for span in selected_spans
                )
                if selected_spans
                else None
            ),
            aligned=bool(selected_spans),
            span_count=len(selected_spans),
            parts=len(value_parts),
            matched_parts=matched_parts,
            fully_raw_matched=(
                matched_parts == len(value_parts)
            ),
            partially_raw_matched=(
                0 < matched_parts < len(value_parts)
            ),
            ambiguous=any(
                count > 1
                for count in candidate_counts
            ),
            candidate_counts=candidate_counts,
            selected_spans=selected_spans,
        )

@dataclass
class EntityReport:
    df: pd.DataFrame

    def build(self) -> pd.DataFrame:
        if self.df.empty:
            return pd.DataFrame()

        result = (
            self.df
            .groupby("entity", as_index=False)
            .agg(
                expected_documents=(
                    "entity",
                    "size",
                ),
                aligned_documents=(
                    "aligned",
                    "sum",
                ),
                fully_raw_matched=(
                    "fully_raw_matched",
                    "sum",
                ),
                partially_raw_matched=(
                    "partially_raw_matched",
                    "sum",
                ),
                ambiguous_documents=(
                    "ambiguous",
                    "sum",
                ),
            )
        )

        result["alignment_coverage"] = (
            result["aligned_documents"]
            / result["expected_documents"]
        )

        result["raw_match_rate"] = (
            result["fully_raw_matched"]
            / result["expected_documents"]
        )

        result["partial_rate"] = (
            result["partially_raw_matched"]
            / result["expected_documents"]
        )

        result["ambiguous_rate"] = (
            result["ambiguous_documents"]
            / result["expected_documents"]
        )

        return result.sort_values(
            "alignment_coverage"
        )

@dataclass
class FinancialReport:
    df: pd.DataFrame

    def build(self) -> pd.DataFrame:
        if self.df.empty:
            return pd.DataFrame()

        financial_details = self.df[
            self.df["entity"].isin(
                [
                    EntityType.TAX.value,
                    EntityType.TOTAL.value,
                ]
            )
        ].copy()

        if financial_details.empty:
            return pd.DataFrame()

        financial_details["value_matches_json"] = (
            financial_details.apply(
                lambda row: (
                    row["observed"] is not None
                    and money_digits(
                        row["observed"]
                    )
                    == money_digits(
                        row["expected"]
                    )
                ),
                axis=1,
            )
        )

        result = (
            financial_details
            .groupby("entity", as_index=False)
            .agg(
                documents=("entity", "size"),
                spans_found=("aligned", "sum"),
                values_matching_json=(
                    "value_matches_json",
                    "sum",
                ),
            )
        )

        result["span_coverage"] = (
            result["spans_found"]
            / result["documents"]
        )

        result["ocr_value_accuracy"] = (
            result["values_matching_json"]
            / result["documents"]
        )

        return result

In [ ]:
from pandas.core.frame import DataFrame

@dataclass(slots=True)
class AlignmentReport:
    documents: pd.DataFrame
    details: pd.DataFrame
    entities: pd.DataFrame
    financial: pd.DataFrame

    @classmethod
    def build(
        cls,
        frame: pd.DataFrame,
        aligner: DocumentAligner,
    ) -> "AlignmentReport":
        document_rows = []
        detail_rows = []

        for row_index, row in frame.iterrows():
            text = str(row["ocred_text"])
            target = cls._get_target(row)
            file_name = row.get(
                "file_name",
                str(row_index),
            )

            expected_fields = tuple(
                target.extract_fields()
            )

            spans = tuple(
                aligner.find_spans(
                    text=text,
                    target=target,
                )
            )

            spans_by_entity = (
                group_spans_by_entity(spans)
            )

            document_rows.append(
                DocumentReportRow.create(
                    row_index=row_index,
                    file_name=file_name,
                    expected_fields=expected_fields,
                    spans=spans,
                )
            )

            detail_rows.extend(
                DetailReportRow.create(
                    row_index=row_index,
                    file_name=file_name,
                    text=text,
                    entity=entity,
                    value=value,
                    selected_spans=(
                        spans_by_entity.get(entity, ())
                    ),
                )
                for entity, value in expected_fields
            )

        documents = pd.DataFrame(
            document_rows
        )

        details = pd.DataFrame(
            detail_rows
        )
    
        entities: DataFrame = EntityReport(df=documents).build()

        financial = FinancialReport(df=documents).build()

        return cls(
            documents=documents,
            details=details,
            entities=entities,
            financial=financial,
        )

    @staticmethod
    def _get_target(
        row: pd.Series,
    ) -> InvoiceTarget:
        target = row.get("target")

        if isinstance(target, InvoiceTarget):
            return target

        if isinstance(target, dict):
            return InvoiceTarget.model_validate(
                target
            )

        json_data = row["json_data"]

        if isinstance(json_data, str):
            return (
                InvoiceTarget
                .model_validate_json(json_data)
            )

        return InvoiceTarget.model_validate(
            json_data
        )

    def missing_counts(self) -> pd.DataFrame:
        missing = (
            self.details.loc[
                ~self.details["aligned"],
                "entity",
            ]
            .value_counts()
            .rename("missing_count")
        )

        return missing.to_frame()

    def low_coverage(
        self,
        limit: int = 10,
    ) -> pd.DataFrame:
        return self.documents.nsmallest(
            limit,
            "alignment_coverage",
        )

## Inspection

In [ ]:
@dataclass(frozen=True, slots=True)
class AlignmentInspector:
    report: AlignmentReport

    def document(
        self,
        row_index: int,
    ) -> pd.DataFrame:
        return (
            self.report.details.loc[
                self.report.details[
                    "row_index"
                ] == row_index
            ]
            .sort_values("entity")
            .reset_index(drop=True)
        )

    def missing(
        self,
    ) -> pd.DataFrame:
        return (
            self.report.details.loc[
                ~self.report.details["aligned"]
            ]
            .sort_values(
                [
                    "entity",
                    "file_name",
                ]
            )
            .reset_index(drop=True)
        )

    def ambiguous(
        self,
    ) -> pd.DataFrame:
        return (
            self.report.details.loc[
                self.report.details["ambiguous"]
            ]
            .sort_values(
                [
                    "entity",
                    "file_name",
                ]
            )
            .reset_index(drop=True)
        )

    def partial_matches(
        self,
    ) -> pd.DataFrame:
        return (
            self.report.details.loc[
                self.report.details[
                    "partially_raw_matched"
                ]
            ]
            .sort_values(
                [
                    "entity",
                    "file_name",
                ]
            )
            .reset_index(drop=True)
        )

    def worst_documents(
        self,
        limit: int = 10,
    ) -> pd.DataFrame:
        return self.report.low_coverage(
            limit
        )

    def financial_mismatches(
        self,
    ) -> pd.DataFrame:
        financial = self.report.details[
            self.report.details["entity"].isin(
                [
                    EntityType.TAX.value,
                    EntityType.TOTAL.value,
                ]
            )
        ].copy()

        financial["value_matches_json"] = (
            financial.apply(
                lambda row: (
                    row["observed"] is not None
                    and money_digits(
                        row["observed"]
                    )
                    == money_digits(
                        row["expected"]
                    )
                ),
                axis=1,
            )
        )

        return (
            financial.loc[
                ~financial[
                    "value_matches_json"
                ]
            ]
            .reset_index(drop=True)
        )

## Tokenizer

In [ ]:
from collections.abc import Mapping
from dataclasses import dataclass
from typing import TypedDict

import numpy as np
from transformers import PreTrainedTokenizerBase

class TokenizedBatch(TypedDict, total=False):
    input_ids: list[list[int]]
    attention_mask: list[list[int]]
    token_type_ids: list[list[int]]
    labels: list[list[int]]


@dataclass(slots=True)
class TokenLabelBatchEncoder:
    """
    Converts invoice OCR text and its expected JSON into the numerical input
    that BERT needs for token classification training

    OCR text + expected JSON =>
    DocumentAligner finds character positions =>
    Tokenizer splits text into BERT tokens =>
    Each token gets 0, B-ENTITY or I-ENTITY =>
    Text tokens and labels become integer arrays
    """

    # Hugging Face tokenizer
    tokenizer: PreTrainedTokenizerBase
    # Finds where expected invoice values occur in the OCR text.
    aligner: DocumentAligner
    # Converts label names into integers
    label2id: Mapping[str, int]
    # BERT can process only a limited number of tokens at once
    # For standard BERT, this is usually 512 tokens.
    # If an invoice contains more than 512 tokens, 
    # it must be divided into multiple chunks.
    max_length: int = 512
    # Chunks overlap by 64 tokens
    window_stride: int = 64

    def __post_init__(self) -> None:
        if self.max_length <= 0:
            raise ValueError(
                "max_length must be positive"
            )

        if not 0 <= self.window_stride < self.max_length:
            raise ValueError(
                "window_stride must be between 0 "
                "and max_length - 1"
            )

        if "O" not in self.label2id:
            raise ValueError(
                "label2id must contain the O label"
            )

    @staticmethod
    def _get_chunk_range(
        offsets: list[tuple[int, int]],
    ) -> tuple[int, int] | None:
        """
        Receives the character offsets for every token in one chunk.
        An offset tells us where a token came from in the original text.

        Example:
            Text: "Invoice 12345"

            Possible offsets:
            [
                (0, 0),    # [CLS] special token
                (0, 7),    # Invoice
                (8, 11),   # 123
                (11, 13),  # ##45
                (0, 0),    # [SEP] special token
            ]

            (0, 7) means the token uses characters 0 through 6.
        """

        valid_offsets = [
            (start, end)
            for start, end in offsets
            if start != end
        ]

        if not valid_offsets:
            return None

        return (
            valid_offsets[0][0],
            valid_offsets[-1][1],
        )

    @staticmethod
    def _calculate_overlap(
        span: EntitySpan,
        chunk_range: tuple[int, int] | None,
    ) -> int:
        """
        Calculates how many characters of an entity occur inside a chunk.
        """

        if chunk_range is None:
            return 0

        chunk_start, chunk_end = chunk_range

        """
        Standard interval-overlap formula.
        Suppose:
            Entity: [80, 120)
            Chunk:  [0, 100)

        The overlapping part is:
            [80, 100)

        Its length is 20 characters.
        The calculation is:
            min(120, 100) - max(80, 0)
        which becomes:
            100 - 80
        which is:
            20
        max(0, ...) prevents a negative result when the ranges do not overlap.
        """
        return max(
            0,
            min(span.end, chunk_end)
            - max(span.start, chunk_start),
        )

    def _find_span_owners(
        self,
        spans: list[EntitySpan],
        chunk_ranges: list[
            tuple[int, int] | None
        ],
    ) -> dict[int, int]:
        """
        Assign each entity span to exactly one token chunk.

        This prevents entities in overlapping tokenizer windows
        from appearing more than once in training/evaluation.
        """

        owners: dict[int, int] = {}

        for span_index, span in enumerate(spans):

            """
            Example:
                Chunk:  [0, 100)
                Entity: [30, 50)
            The chunk fully contains the entity.
            But:
                Chunk:  [0, 100)
                Entity: [80, 120)

            does not fully contain it.
            """
            containing_chunks = [
                chunk_index
                for chunk_index, chunk_range
                in enumerate(chunk_ranges)
                if (
                    chunk_range is not None
                    and chunk_range[0] <= span.start
                    and span.end <= chunk_range[1]
                )
            ]

            # If one or more chunks fully contain the entity,
            # assign it to the first one.
            # It might appear completely in two 
            # chunks because of the stride overlap.
            # Choosing the first prevents duplicate training labels.
            if containing_chunks:
                owners[span_index] = (
                    containing_chunks[0]
                )
                continue
            
            # If no chunk completely contains the entity, 
            # calculate how much of it appears in every chunk.
            overlaps = [
                self._calculate_overlap(
                    span,
                    chunk_range,
                )
                for chunk_range in chunk_ranges
            ]

            # no chunks => nowhere to assign the entity.
            if not overlaps:
                continue

            best_chunk = int(np.argmax(overlaps))

            # Only assign the entity 
            # if the selected chunk actually overlaps it.
            if overlaps[best_chunk] > 0:
                owners[span_index] = best_chunk

        return owners

    def _label_chunk(
        self,
        offsets: list[tuple[int, int]],
        chunk_index: int,
        spans: list[EntitySpan],
        span_owners: Mapping[int, int],
    ) -> list[int]:
        """
        This method produces one label ID for every token in one chunk.
        For example:
            [0, 0, 1, 2, -100]
        could mean:
            Token 1 → O
            Token 2 → O
            Token 3 → B-INVOICE_NUMBER
            Token 4 → I-INVOICE_NUMBER
            Token 5 → ignored
        """

        labels: list[int] = []
        started_spans: set[int] = set()

        # Process every token’s character range.
        for token_start, token_end in offsets:
            # Special and padding tokens.
            # A token with an empty range is usually:
            # * `[CLS]`
            # * `[SEP]`
            # * Padding
            if token_start == token_end:
                labels.append(-100)
                continue

            overlapping_span: (
                tuple[int, EntitySpan] | None
            ) = None

            for span_index, span in enumerate(spans):
                # Check whether the token range and entity range overlap.
                overlaps = (
                    token_start < span.end
                    and token_end > span.start
                )
                # Store the first matching entity and stop searching.
                # This is safe because InvoiceSpanAligner
                # prevents entity spans from overlapping each other.
                if overlaps:
                    overlapping_span = (
                        span_index,
                        span,
                    )
                    break
            
            # If the token is not inside an entity
            # give it the `O` label
            if overlapping_span is None:
                labels.append(
                    self.label2id["O"]
                )
                continue

            span_index, span = overlapping_span

            # Ignore duplicate span copies produced by
            # overlapping tokenizer windows.
            # The entity may appear in multiple overlapping chunks.
            # If this chunk is not the entity’s selected owner, ignore its token with -100.
            # This avoids counting or training on the entity twice.
            if span_owners.get(span_index) != chunk_index:
                labels.append(-100)
                continue

            prefix = (
                "B"
                if span_index not in started_spans
                else "I"
            )

            label_name = (
                f"{prefix}-{span.entity}"
            )

            # Check that the label exists in the configured label mapping.
            # This catches situations where you add a new entity to InvoiceTarget
            # but forget to add its BIO labels to label2id.
            if label_name not in self.label2id:
                # Stop with an informative error rather than 
                # silently assigning an incorrect label.
                raise KeyError(
                    f"Missing label in label2id: "
                    f"{label_name}"
                )

            # Convert the label string to a number and 
            # add it to the result.
            labels.append(
                self.label2id[label_name]
            )
            started_spans.add(span_index)

        return labels

    @staticmethod
    def _parse_target(
        value: object,
    ) -> InvoiceTarget:
        """
        Handles three possible input formats: 
        * InvoiceTarget
        * str
        * dict
        """

        if isinstance(value, InvoiceTarget):
            return value

        if isinstance(value, str):
            return InvoiceTarget.model_validate_json(
                value
            )

        return InvoiceTarget.model_validate(
            value
        )

    def __call__(
        self,
        examples: Mapping[str, list],
    ) -> TokenizedBatch:
        """
        Makes the object callable for passing it to Hugging Face
        """
        
        """
        The tokenizer declares the inputs expected by its model.
        For BERT, these are commonly:
            [
                "input_ids",
                "token_type_ids",
                "attention_mask",
            ]
        The code creates:
            {
                "input_ids": [],
                "token_type_ids": [],
                "attention_mask": [],
            }
        """
        output: TokenizedBatch = {
            "input_ids": [],
            "token_type_ids": [],
            "attention_mask": [],
            "labels": []
        }

        # Processing every document in the batch
        for text, json_data in zip(
            examples["ocred_text"],
            examples["json_data"],
        ):
            text = str(text)

            target = self._parse_target(
                value=json_data
            )

            # Finding entity character spans
            # Connects the target JSON values with 
            # their locations in the OCR text.
            # At this point, labels exist at the character level
            # They are not yet token labels.
            spans = self.aligner.find_spans(
                text=text,
                target=target,
            )
            
            encoding = self.tokenizer(
                # Pass the OCR text into the Hugging Face tokenizer
                text,
                # The tokenizer must respect `max_length`.
                # Because overflow handling is enabled, 
                # text beyond the first chunk is placed into additional 
                # chunks rather than simply discarded.
                truncation=True,
                # Each chunk can contain at most this many tokens.
                max_length=self.max_length,
                # Neighbouring chunks overlap by this many tokens.
                stride=self.window_stride,
                # Create multiple chunks when the document is too long.
                # Without this, the text beyond `max_length` would be lost.
                return_overflowing_tokens=True,
                # Return each token’s original character location.
                # For example:
                #   "12345"
                # might become:
                #   "123"   → (16, 19)
                #   "##45"  → (19, 21)
                # These offsets are what connect character spans 
                # from `InvoiceSpanAligner` to BERT tokens.
                return_offsets_mapping=True,
            )

            # Remove offsets from `encoding` and save them separately.
            # Offsets are needed to build labels,
            # but they are not an input to BERT.
            offset_chunks = encoding.pop(
                "offset_mapping"
            )

            # The tokenizer can return metadata showing which original example produced each overflow chunk.
            # This encoder processes one document at a time inside the loop, so that metadata is unnecessary.
            # The second argument, `None`, means “do not raise an error if this key does not exist.”
            encoding.pop(
                "overflow_to_sample_mapping",
                None,
            )

            # Finding chunk character ranges
            # For each chunk, calculate the portion of the original text it covers.
            # Example:
            #   [
            #       (0, 1800),
            #       (1550, 3200),
            #   ]
            # The overlap from characters 1550–1800 comes from the tokenizer stride.
            chunk_ranges = [
                self._get_chunk_range(offsets)
                for offsets in offset_chunks
            ]

            # Selecting one owner per entity
            # Choose exactly one chunk responsible for each entity.
            # Example:
            #   {
            #       0: 0,
            #       1: 0,
            #       2: 1,
            #   }
            # This prevents duplicate entity labels in overlapping chunks.
            span_owners = self._find_span_owners(
                spans=spans,
                chunk_ranges=chunk_ranges,
            )

            # Creating labels for each chunk
            # Process every chunk created from this document.
            for chunk_index, offsets in enumerate(
                offset_chunks
            ):
                # Create one numerical label per token in this chunk.
                # Example:
                #   [-100, 0, 0, 1, 2, -100]
                labels = self._label_chunk(
                    offsets=offsets,
                    chunk_index=chunk_index,
                    spans=spans,
                    span_owners=span_owners,
                )

                # Saving model inputs
                # Process each input required by BERT:
                # * input_ids
                # * attention_mask
                # * token_type_ids

                for name in (
                    self.tokenizer.model_input_names
                ):
                    # Take the input values for the current chunk and 
                    # add them to the output batch.
                    output[name].append(
                        encoding[name][chunk_index]
                    )

                # Add the labels belonging to the same chunk.
                # The positions line up:
                #   input_ids[0][5]
                # and:
                #   labels[0][5]
                # refer to the same token.

                output["labels"].append(labels)

        # Return all generated chunks and labels.
        # The result looks approximately like:
        # {
        #     "input_ids": [
        #         [101, 1999, 6764, 102, ...],
        #         [101, 2023, 2003, 102, ...],
        #     ],
        #     "attention_mask": [
        #         [1, 1, 1, 1, ...],
        #         [1, 1, 1, 1, ...],
        #     ],
        #     "token_type_ids": [
        #         [0, 0, 0, 0, ...],
        #         [0, 0, 0, 0, ...],
        #     ],
        #     "labels": [
        #         [-100, 0, 1, -100, ...],
        #         [-100, 0, 0, -100, ...],
        #     ],
        # }
        # Each row represents one chunk, not necessarily one original invoice.
        # One long invoice can produce several rows.
        return output

### Test

In [ ]:
batch_encoder = TokenLabelBatchEncoder(
    tokenizer=tokenizer,
    aligner=document_aligner,
    label2id=label2id,
    max_length=MAX_LENGTH,
    window_stride=WINDOW_STRIDE,
)

tokenized_datasets = raw_datasets.map(
    batch_encoder,
    batched=True,
    batch_size=16,
    remove_columns=[
        "ocred_text",
        "json_data",
    ],
    load_from_cache_file=False,
)

## Trainer

In [194]:
def get_nested(data, *keys):
    value = data

    for key in keys:
        if not isinstance(value, dict):
            return None

        value = value.get(key)

    return value

def extract_phase_one_fields(target):
    fields = [
        (
            "INVOICE_NUMBER",
            get_nested(
                target,
                "invoice",
                "invoice_number",
            ),
        ),
        (
            "INVOICE_DATE",
            get_nested(
                target,
                "invoice",
                "invoice_date",
            ),
        ),
        (
            "INVOICE_DUE_DATE",
            get_nested(
                target,
                "invoice",
                "due_date",
            ),
        ),
        (
            "SELLER_NAME",
            get_nested(
                target,
                "invoice",
                "seller_name",
            ),
        ),
        (
            "SELLER_ADDRESS",
            get_nested(
                target,
                "invoice",
                "seller_address",
            ),
        ),
        (
            "CLIENT_NAME",
            get_nested(
                target,
                "invoice",
                "client_name",
            ),
        ),
        (
            "CLIENT_ADDRESS",
            get_nested(
                target,
                "invoice",
                "client_address",
            ),
        ),
        (
            "TAX",
            get_nested(
                target,
                "subtotal",
                "tax",
            ),
        ),
        (
            "DISCOUNT",
            get_nested(
                target,
                "subtotal",
                "discount",
            ),
        ),
        (
            "TOTAL",
            get_nested(
                target,
                "subtotal",
                "total",
            ),
        ),
        (
            "PAYMENT_DUE_DATE",
            get_nested(
                target,
                "payment_instructions",
                "due_date",
            ),
        ),
        (
            "BANK_NAME",
            get_nested(
                target,
                "payment_instructions",
                "bank_name",
            ),
        ),
        (
            "ACCOUNT_NUMBER",
            get_nested(
                target,
                "payment_instructions",
                "account_number",
            ),
        ),
        (
            "PAYMENT_METHOD",
            get_nested(
                target,
                "payment_instructions",
                "payment_method",
            ),
        ),
    ]

    return [
        (entity, str(value).strip())
        for entity, value in fields
        if value is not None and str(value).strip()
    ]

In [103]:
row = df.iloc[0]

text = row["ocred_text"]
target = json.loads(row["json_data"])

print(text)
print(extract_baseline_fields(target))

Invoice no: 84652373 Date of issue: 02/23/2021 Seller: Client: Nguyen-Roach Clark-Foster 247 David Highway 77477 Cliff Apt. 853 Lake John, WV 84178 Washingtonbury, MS 78346 Tax Id: 991-72-5826 Tax Id: 937-70-8530 IBAN: GB91/YXO05542456978150 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth Stemware Rack Display Kitchen 1,00 each 42,32 42,32 10% 46,55 Wine Glass Holder Bottle Carbon Steel Free Punch 2 VTG (4) 7 Ounce Since 1852 1,00 each 14,00 14,00 10% 15,40 Milk Bottle Wine Carafe Juice Glass with Cork Lids 3 Vintage Crystal Red Wine 1,00 each 35,45 35,45 10% 39,00 Glasses NOS West Germany 1983 6 10 ounce elegant stems 3 Ikea Stainless Steel 4-bottle 4,00 each 25,00 100,00 10% 110,00 Wine Rack 300.557.60 great condition gift it! 5 Lolita "Wine Bouquet" Hand 1,00 each 20,00 20,00 10% 22,00 Painted and Decorated Wine Glass NIB SUMMARY VAT [%] Net worth VAT Gross worth 10% 211,77 21,18 232,95 Total $ 211,77 $ 21,18 $ 232,95 Troy
[('INVOICE_NUMBER', '84652373'), ('INVO

In [195]:
def normalize_with_mapping(text):
    normalized = []
    original_positions = []

    previous_was_space = True

    for index, character in enumerate(str(text)):
        if character.isalnum():
            normalized.append(character.lower())
            original_positions.append(index)
            previous_was_space = False

        elif not previous_was_space:
            normalized.append(" ")
            original_positions.append(index)
            previous_was_space = True

    if normalized and normalized[-1] == " ":
        normalized.pop()
        original_positions.pop()

    return "".join(normalized), original_positions


def normalize_for_matching(value):
    normalized, _ = normalize_with_mapping(value)
    return normalized

def find_normalized_matches(text, value):
    normalized_text, positions = normalize_with_mapping(text)
    normalized_value = normalize_for_matching(value)

    if not normalized_value:
        return []

    matches = []
    start = 0

    while True:
        normalized_start = normalized_text.find(
            normalized_value,
            start,
        )

        if normalized_start == -1:
            break

        normalized_end = (
            normalized_start + len(normalized_value)
        )

        original_start = positions[normalized_start]
        original_end = positions[normalized_end - 1] + 1

        matches.append(
            (original_start, original_end)
        )

        start = normalized_start + 1

    return matches

In [196]:
def get_value_parts(entity, value):
    if entity in {
        "SELLER_ADDRESS",
        "CLIENT_ADDRESS",
    }:
        return [
            part.strip()
            for part in str(value).splitlines()
            if part.strip()
        ]

    return [str(value)]

In [197]:
def extract_baseline_fields(target):
    return [
        (entity, value)
        for entity, value
        in extract_phase_one_fields(target)
        if entity in BASELINE_ENTITIES
    ]

In [ ]:
def select_entity_candidate(text, entity, candidates):
    if not candidates:
        return None

    if entity in {"TAX", "TOTAL"}:
        # Prefer values appearing after the final "Total" label.
        total_anchors = [
            match.start()
            for match in re.finditer(
                r"\btotal\b",
                text,
                flags=re.IGNORECASE,
            )
        ]

        if total_anchors:
            final_total_anchor = total_anchors[-1]

            candidates_after_total = [
                candidate
                for candidate in candidates
                if candidate[0] > final_total_anchor
            ]

            if candidates_after_total:
                return candidates_after_total[-1]

        # Fallback for documents without a recognizable Total label.
        return candidates[-1]

    if len(candidates) == 1:
        return candidates[0]

    return None

In [199]:
def find_numeric_matches(text, value):
    digits = "".join(
        re.findall(r"\d", str(value))
    )

    if not digits:
        return []

    pattern = (
        r"(?<!\d)"
        + r"[\s,.'’]*".join(
            re.escape(digit)
            for digit in digits
        )
        + r"(?!\d)"
    )

    return [
        match.span()
        for match in re.finditer(pattern, text)
    ]

In [200]:
def find_identifier_matches(text, value):
    characters = re.findall(
        r"[a-zA-Z0-9]",
        str(value),
    )

    if not characters:
        return []

    pattern = (
        r"(?<![a-zA-Z0-9])"
        + r"[^a-zA-Z0-9]*".join(
            re.escape(character)
            for character in characters
        )
        + r"(?![a-zA-Z0-9])"
    )

    return [
        match.span()
        for match in re.finditer(
            pattern,
            text,
            flags=re.IGNORECASE,
        )
    ]

In [201]:
def find_value_matches(text, entity, value):
    if entity in {
        "TAX",
        "DISCOUNT",
        "TOTAL",
    }:
        return find_numeric_matches(text, value)

    if entity == "ACCOUNT_NUMBER":
        return find_identifier_matches(text, value)

    return find_normalized_matches(text, value)

In [202]:
MONEY_PATTERN = re.compile(
    r"""
    (?<![\dA-Za-z])
    (
        (?:
            \d{1,3}
            (?:[ \u00A0.'’]\d{3})+
            |
            \d+
        )
        [,.]
        \d{2}
    )
    (?!\d)
    """,
    flags=re.VERBOSE,
)


def find_final_total_row(text, lookahead=200):
    """
    Expected final row:

        Total $ net $ tax $ gross

    Returns character spans from the observed OCR text.
    """
    total_anchors = list(
        re.finditer(
            r"\btotal\b",
            text,
            flags=re.IGNORECASE,
        )
    )

    # Search backwards in case "total" occurs elsewhere.
    for anchor in reversed(total_anchors):
        region_end = min(
            len(text),
            anchor.end() + lookahead,
        )

        amounts = list(
            MONEY_PATTERN.finditer(
                text,
                anchor.end(),
                region_end,
            )
        )

        if len(amounts) < 3:
            continue

        net_match = amounts[0]
        tax_match = amounts[1]
        total_match = amounts[2]

        return {
            "NET": net_match.span(),
            "TAX": tax_match.span(),
            "TOTAL": total_match.span(),
        }

    return {}

In [203]:
row = df.loc[
    df["file_name"] == "batch1-0028.jpg"
].iloc[0]

financial_spans = find_final_total_row(
    row["ocred_text"]
)

for entity, (start, end) in financial_spans.items():
    print(
        entity,
        repr(row["ocred_text"][start:end]),
    )

NET '1 036,58'
TAX '103,66'
TOTAL '1 140,24'


In [204]:
from dataclasses import dataclass

def value_pattern(value):
    parts = re.split(r"\s+", str(value).strip())

    return r"\s+".join(
        re.escape(part)
        for part in parts
    )

def find_entity_spans(text, target) -> list[EntitySpan]:
    text = str(text)
    financial_spans = find_final_total_row(text)

    if isinstance(target, str):
        target = json.loads(target)

    # Get baseline (eg fields that are expected) fields in format
    # [("<field name>", "field value")]
    fields = extract_baseline_fields(target)

    spans: list[EntitySpan] = []
    occupied = []
    for entity, value in fields:
        # Return values for given fields in format
        # [("value")]
        value_parts = get_value_parts(
            entity,
            value,
        )

        for value_part in value_parts:
            # For tax and total, the final summary-row position is
            # more reliable than matching repeated values.
            if entity in financial_spans:
                candidates = [
                    financial_spans[entity]
                ]
            else:
                # Return sequence of indexes for value part that appeared in the text
                # [(<start index>, <end index>)]
                candidates = find_value_matches(
                    text,
                    entity,
                    value_part,
                )

            available = []

            for start, end in candidates:
                overlaps = any(
                    start < occupied_end
                    and end > occupied_start
                    for occupied_start, occupied_end
                    in occupied
                )

                if not overlaps:
                    available.append((start, end))

            if entity in financial_spans:
                selected = (
                    available[0]
                    if available
                    else None
                )
            else:
                # Special case for "TAX" and "TOTAL" values
                # If it is not one of them - returns first candidate
                selected = select_entity_candidate(
                    text,
                    entity,
                    available,
                )

            if selected is None:
                continue

            start, end = selected
            
            # Form final response based on the foundede indexes
            spans.append(
                EntitySpan(
                    start= start,
                    end= end,
                    entity= entity,
                    # Store the OCR-observed value, even if OCR made an error.
                    value=text[start:end]
                )
            )

            occupied.append((start, end))

    # return appeared entities in the ascending order
    return sorted(
        spans,
        key=lambda span: span["start"],
    )

#### Test

In [205]:
row = df.iloc[10]
text = row["ocred_text"]
target = json.loads(row["json_data"])

In [206]:
fields = extract_baseline_fields(target)
fields

[('INVOICE_NUMBER', '72591450'),
 ('INVOICE_DATE', '05/13/2012'),
 ('SELLER_NAME', 'Bishop, Kelley and Wright'),
 ('SELLER_ADDRESS', '058 Sullivan Village\nNorth Isabella, ND 69747'),
 ('CLIENT_NAME', 'Allen, Murray and Carroll'),
 ('CLIENT_ADDRESS', '87328 Arnold Crossroad Suite 206\nSouth Amy, MS 29781'),
 ('TAX', '480,29'),
 ('TOTAL', '5 283,18')]

In [207]:
find_entity_spans(text=text, target=target)

[{'start': 12, 'end': 20, 'entity': 'INVOICE_NUMBER', 'value': '72591450'},
 {'start': 36, 'end': 46, 'entity': 'INVOICE_DATE', 'value': '05/13/2012'},
 {'start': 63,
  'end': 88,
  'entity': 'SELLER_NAME',
  'value': 'Bishop, Kelley and Wright'},
 {'start': 89,
  'end': 114,
  'entity': 'CLIENT_NAME',
  'value': 'Allen, Murray and Carroll'},
 {'start': 115,
  'end': 135,
  'entity': 'SELLER_ADDRESS',
  'value': '058 Sullivan Village'},
 {'start': 136,
  'end': 168,
  'entity': 'CLIENT_ADDRESS',
  'value': '87328 Arnold Crossroad Suite 206'},
 {'start': 169,
  'end': 193,
  'entity': 'SELLER_ADDRESS',
  'value': 'North Isabella, ND 69747'},
 {'start': 1162, 'end': 1168, 'entity': 'TAX', 'value': '480,29'},
 {'start': 1171, 'end': 1179, 'entity': 'TOTAL', 'value': '5 283,18'}]

In [208]:
entity, value = fields[0]

value_part = get_value_parts(
    entity,
    value,
)
value_part

['72591450']

In [209]:
normalize_for_matching(value_part)

'72591450'

In [210]:
normalize_with_mapping(text)

('invoice no 72591450 date of issue 05 13 2012 seller client bishop kelley and wright allen murray and carroll 058 sullivan village 87328 arnold crossroad suite 206 north isabella nd 69747 south ms 29781 tax id 956 72 8059 tax id 902 99 1117 iban gb63muda41776263491543 items no description qty um net price net worth vat gross worth dell optiplex desktop computer 5 00 each 149 95 749 75 10 824 73 with dual 19 lcd monitors windows 10 pro 2 fast dell optiplex windows 10 3 00 each 119 95 359 85 10 395 84 pro desktop computer dual core 6gb 160gb dvd wifi 3 hp prodesk 400 gs mini 5 00 each 200 50 1 002 50 10 1 102 75 desktop computer i5 9500t 8gb 256gb ssd wiop custom gaming computer 2 00 each 450 00 900 00 10 990 00 5 fast dell optiplex windows 10 3 00 each 124 95 374 85 10 412 34 desktop computer tower czd 4gb dvd wifi 17 lcd amd dual core 3 9ghz 4gb 3 00 each 249 99 749 97 10 824 97 ddr3 500gb wifi windows 10 office desktop pc computer custom build dell optiplex 9020 3 00 each 221 99 665 

In [211]:
find_normalized_matches(
    text,
    value_part,
)

[(12, 20)]

In [212]:
text[12:20]

'72591450'

In [213]:
select_entity_candidate(
    text=text,
    entity=entity,
    candidates=[(12,20)]
)

(12, 20)

In [214]:
def money_key(value):
    return "".join(
        re.findall(r"\d", str(value))
    )


financial_quality = []

for _, row in df.iterrows():
    target = row["target"]
    expected = dict(
        extract_baseline_fields(target)
    )

    detected = find_final_total_row(
        row["ocred_text"]
    )

    for entity in ["TAX", "TOTAL"]:
        expected_value = expected.get(entity)
        detected_span = detected.get(entity)

        if detected_span is None:
            observed_value = None
        else:
            start, end = detected_span
            observed_value = row["ocred_text"][start:end]

        financial_quality.append({
            "entity": entity,
            "file_name": row["file_name"],
            "span_found": detected_span is not None,
            "expected": expected_value,
            "observed": observed_value,
            "value_matches": (
                detected_span is not None
                and money_key(expected_value)
                == money_key(observed_value)
            ),
        })


financial_quality_df = pd.DataFrame(
    financial_quality
)

financial_summary = (
    financial_quality_df
    .groupby("entity")
    .agg(
        documents=("file_name", "size"),
        spans_found=("span_found", "sum"),
        values_matching=("value_matches", "sum"),
    )
)

financial_summary["span_coverage"] = (
    financial_summary["spans_found"]
    / financial_summary["documents"]
)

financial_summary["ocr_value_accuracy"] = (
    financial_summary["values_matching"]
    / financial_summary["documents"]
)


financial_summary.style.format({
    "span_coverage": "{:.2%}",
    "ocr_value_accuracy": "{:.2%}",
})

,documents,spans_found,values_matching,span_coverage,ocr_value_accuracy
entity,,,,,
TAX,1413,1413,1409,100.00%,99.72%
TOTAL,1413,1413,1385,100.00%,98.02%


### Calculate the new alignment 

In [215]:
from collections import Counter

expected_documents = Counter()
matched_documents = Counter()

for _, row in df.iterrows():
    target = row["target"]

    expected_entities = {
        entity
        for entity, value in extract_baseline_fields(target)
    }

    spans = find_entity_spans(
        row["ocred_text"],
        target,
    )

    matched_entities = {
        span["entity"]
        for span in spans
    }

    for entity in expected_entities:
        expected_documents[entity] += 1

    for entity in expected_entities & matched_entities:
        matched_documents[entity] += 1


alignment_report = pd.DataFrame([
    {
        "entity": entity,
        "expected_documents": expected_documents[entity],
        "matched_documents": matched_documents[entity],
        "coverage": (
            matched_documents[entity]
            / expected_documents[entity]
        ),
    }
    for entity in sorted(expected_documents)
])

alignment_report.style.format({
    "coverage": "{:.2%}",
})

,entity,expected_documents,matched_documents,coverage
0,CLIENT_ADDRESS,1413,1413,100.00%
1,CLIENT_NAME,1413,1395,98.73%
2,INVOICE_DATE,1413,1413,100.00%
3,INVOICE_NUMBER,1413,1413,100.00%
4,SELLER_ADDRESS,1413,1413,100.00%
5,SELLER_NAME,1413,1400,99.08%
6,TAX,1413,1413,100.00%
7,TOTAL,1413,1413,100.00%


### Manually inspect spans

In [216]:
sample = df.sample(
    50,
    random_state=42,
)

for _, row in sample.iterrows():
    spans = find_entity_spans(
        row["ocred_text"],
        row["target"],
    )

    financial_spans = [
        span
        for span in spans
        if span["entity"] in {"TAX", "TOTAL"}
    ]

    print(row["file_name"])

    for span in financial_spans:
        context_start = max(0, span["start"] - 60)
        context_end = span["end"] + 60

        print(
            span["entity"],
            "=>",
            repr(row["ocred_text"][context_start:context_end]),
        )

    print("=" * 100)

batch1-0897.jpg
TAX => 'th VAT Gross worth 10% 310,57 31,06 341,63 Total $ 310,57 $ 31,06 $ 341,63'
TOTAL => 'ross worth 10% 310,57 31,06 341,63 Total $ 310,57 $ 31,06 $ 341,63'
batch1-0896.jpg
TAX => 'th VAT Gross worth 10% 889,67 88,97 978,64 Total $ 889,67 $ 88,97 $ 978,64 Sexy'
TOTAL => 'ross worth 10% 889,67 88,97 978,64 Total $ 889,67 $ 88,97 $ 978,64 Sexy'
batch1-0412.jpg
TAX => 'th VAT Gross worth 10% 258,96 25,90 284,86 Total $ 258,96 $ 25,90 $ 284,86'
TOTAL => 'ross worth 10% 258,96 25,90 284,86 Total $ 258,96 $ 25,90 $ 284,86'
batch1-0028.jpg
TAX => 'Gross worth 10% 1 036,58 103,66 1 140,24 Total $ 1 036,58 $ 103,66 $ 1 140,24 Sega'
TOTAL => 'th 10% 1 036,58 103,66 1 140,24 Total $ 1 036,58 $ 103,66 $ 1 140,24 Sega'
batch1-0025.jpg
TAX => 'Gross worth 10% 7 200,00 720,00 7 920,00 Total $ 7 200,00 $ 720,00 $ 7 920,00'
TOTAL => 'th 10% 7 200,00 720,00 7 920,00 Total $ 7 200,00 $ 720,00 $ 7 920,00'
batch1-0851.jpg
TAX => 'Gross worth 10% 4 749,97 475,00 5 224,97 Total $ 4 749,9

In [217]:
def bio_to_entities(labels):
    entities = []
    current_type = None
    current_start = None

    for index, label in enumerate(labels + ["O"]):
        if label == "O":
            if current_type is not None:
                entities.append(
                    (
                        current_type,
                        current_start,
                        index,
                    )
                )
                current_type = None
                current_start = None

            continue

        prefix, entity_type = label.split("-", 1)

        starts_new = (
            prefix == "B"
            or current_type is None
            or current_type != entity_type
        )

        if starts_new:
            if current_type is not None:
                entities.append(
                    (
                        current_type,
                        current_start,
                        index,
                    )
                )

            current_type = entity_type
            current_start = index

    return set(entities)

In [218]:
row = df.iloc[0]

text = row["ocred_text"]
target = json.loads(row["json_data"])

spans = find_entity_spans(text, target)

for span in spans:
    print(span)

{'start': 12, 'end': 20, 'entity': 'INVOICE_NUMBER', 'value': '84652373'}
{'start': 36, 'end': 46, 'entity': 'INVOICE_DATE', 'value': '02/23/2021'}
{'start': 63, 'end': 75, 'entity': 'SELLER_NAME', 'value': 'Nguyen-Roach'}
{'start': 76, 'end': 88, 'entity': 'CLIENT_NAME', 'value': 'Clark-Foster'}
{'start': 89, 'end': 106, 'entity': 'SELLER_ADDRESS', 'value': '247 David Highway'}
{'start': 128, 'end': 147, 'entity': 'SELLER_ADDRESS', 'value': 'Lake John, WV 84178'}
{'start': 148, 'end': 172, 'entity': 'CLIENT_ADDRESS', 'value': 'Washingtonbury, MS 78346'}
{'start': 941, 'end': 946, 'entity': 'TAX', 'value': '21,18'}
{'start': 949, 'end': 955, 'entity': 'TOTAL', 'value': '232,95'}


In [219]:
alignment_records = []

for _, row in df.iterrows():
    target = json.loads(row["json_data"])
    expected = extract_baseline_fields(target)
    matched = find_entity_spans(
        row["ocred_text"],
        target,
    )

    alignment_records.append({
        "expected_fields": len(expected),
        "matched_fields": len(matched),
        "coverage": (
            len(matched) / len(expected)
            if expected
            else 1.0
        ),
    })

alignment_df = pd.DataFrame(alignment_records)

alignment_df.describe()

,expected_fields,matched_fields,coverage
count,1413.0,1413.000000,1413.000000
mean,8.0,9.910828,1.238854
std,0.0,0.294862,0.036858
min,8.0,8.000000,1.000000
25%,8.0,10.000000,1.250000
50%,8.0,10.000000,1.250000
75%,8.0,10.000000,1.250000
max,8.0,10.000000,1.250000


In [220]:
low_coverage_indices = (
    alignment_df
    .nsmallest(10, "coverage")
    .index
)

for index in low_coverage_indices:
    row = df.iloc[index]
    target = json.loads(row["json_data"])

    print("Coverage:", alignment_df.loc[index, "coverage"])
    print("Expected:", extract_phase_one_fields(target))
    print("Matched:", find_entity_spans(
        row["ocred_text"],
        target,
    ))
    print("=" * 100)

Coverage: 1.0
Expected: [('INVOICE_NUMBER', '91296589'), ('INVOICE_DATE', '09/04/2019'), ('SELLER_NAME', 'Wagner, Jensen and Key'), ('SELLER_ADDRESS', '2023 Morgan Islands\nLake Jenniferland, IL 67123'), ('CLIENT_NAME', 'King Inc'), ('CLIENT_ADDRESS', '78997 Brent Court\nJonesland, ΜΑ 32167'), ('TAX', '151,80'), ('TOTAL', '1 669,75')]
Matched: [{'start': 12, 'end': 20, 'entity': 'INVOICE_NUMBER', 'value': '91296589'}, {'start': 36, 'end': 46, 'entity': 'INVOICE_DATE', 'value': '09/04/2019'}, {'start': 82, 'end': 90, 'entity': 'CLIENT_NAME', 'value': 'King Inc'}, {'start': 91, 'end': 110, 'entity': 'SELLER_ADDRESS', 'value': '2023 Morgan Islands'}, {'start': 111, 'end': 128, 'entity': 'CLIENT_ADDRESS', 'value': '78997 Brent Court'}, {'start': 129, 'end': 156, 'entity': 'SELLER_ADDRESS', 'value': 'Lake Jenniferland, IL 67123'}, {'start': 609, 'end': 615, 'entity': 'TAX', 'value': '151,80'}, {'start': 618, 'end': 626, 'entity': 'TOTAL', 'value': '1 669,75'}]
Coverage: 1.0
Expected: [('INV

In [221]:
def calculate_alignment(row):
    target = json.loads(row["json_data"])

    expected = extract_baseline_fields(target)
    matched = find_entity_spans(
        row["ocred_text"],
        target,
    )

    expected_entities = {
        entity
        for entity, value in expected
    }

    matched_entities = {
        span["entity"]
        for span in matched
    }

    missing_entities = sorted(
        expected_entities - matched_entities
    )

    return pd.Series({
        "expected_entities": len(expected_entities),
        "matched_entities": len(
            expected_entities & matched_entities
        ),
        "alignment_coverage": (
            len(expected_entities & matched_entities)
            / len(expected_entities)
            if expected_entities
            else 1.0
        ),
        "missing_entities": missing_entities,
    })


In [222]:
alignment_stats = df.apply(
    calculate_alignment,
    axis=1,
)

annotated_df = df.drop(
    columns=[
        "expected_entities",
        "matched_entities",
        "alignment_coverage",
        "missing_entities",
    ],
    errors="ignore",
)

annotated_df = pd.concat(
    [annotated_df, alignment_stats],
    axis=1,
)

### Debug missing fields

In [92]:
def find_numeric_matches(text, value):
    digits = "".join(
        re.findall(r"\d", str(value))
    )

    if not digits:
        return []

    pattern = (
        r"(?<!\d)"
        + r"[\s,.'’]*".join(
            re.escape(digit)
            for digit in digits
        )
        + r"(?!\d)"
    )

    return [
        match.span()
        for match in re.finditer(pattern, text)
    ]

In [93]:
print(
    find_numeric_matches(
        "Tax: 101,51 Total: 1 116,62",
        "101.51",
    )
)

print(
    find_numeric_matches(
        "Tax: 101,51 Total: 1 116,62",
        "1116.62",
    )
)

[(5, 11)]
[(19, 27)]


In [94]:
def find_identifier_matches(text, value):
    characters = re.findall(
        r"[a-zA-Z0-9]",
        str(value),
    )

    if not characters:
        return []

    pattern = (
        r"(?<![a-zA-Z0-9])"
        + r"[^a-zA-Z0-9]*".join(
            re.escape(character)
            for character in characters
        )
        + r"(?![a-zA-Z0-9])"
    )

    return [
        match.span()
        for match in re.finditer(
            pattern,
            text,
            flags=re.IGNORECASE,
        )
    ]

In [95]:
print(
    find_identifier_matches(
        "Account: GB75 EHJO 7174 7105 5734 03",
        "GB75EHJO71747105573403",
    )
)

[(9, 36)]


In [96]:
def find_value_matches(text, entity, value):
    if entity in {
        "TAX",
        "DISCOUNT",
        "TOTAL",
    }:
        return find_numeric_matches(text, value)

    if entity == "ACCOUNT_NUMBER":
        return find_identifier_matches(text, value)

    return find_normalized_matches(text, value)

In [97]:
def debug_missing_fields(row):
    text = row["ocred_text"]
    target = json.loads(row["json_data"])

    spans = find_entity_spans(text, target)

    matched_entities = {
        span["entity"]
        for span in spans
    }

    for entity, value in extract_phase_one_fields(target):
        if entity in matched_entities:
            continue

        matches = find_value_matches(
            text,
            entity,
            value,
        )

        print({
            "entity": entity,
            "expected": value,
            "matches_count": len(matches),
            "matches": [
                text[start:end]
                for start, end in matches
            ],
        })

In [223]:
row = annotated_df[
    annotated_df["json_data"].str.contains(
        "92944130",
        regex=False,
    )
].iloc[0]
row

,612
file_name,batch1-0689.jpg
json_data,"\n{\n ""invoice"": {\n ""client_name"": ""Long,..."
ocred_text,Invoice no: 92944130 Date of issue: 07/07/2014...
subdir,batch1_2
source_group,batch_1/batch_1
csv_path,/content/invoices/batch_1/batch_1/batch1_2.csv
path,/content/invoices/batch_1/batch_1/batch1_2/bat...
target,{'invoice': {'client_address': '204 Ward Harbo...
top_level_keys,"[invoice, items, payment_instructions, subtotal]"
target_text,"{""invoice"":{""client_address"":""204 Ward Harbors..."


In [99]:
text = row["ocred_text"]

print(repr(text[:300]))

'Invoice no: 92944130 Date of issue: 07/07/2014 Seller: Client: Fritz , and Gomez Long, Jacobson and Roman 9654 Mary 204 Ward Harbors Jacksonhaven, DE 27036 New Adam, AR 28820 Tax Id: 922-99-1658 Tax Id: 989-78-8644 IBAN: GBOGVHPH54755658499923 ITEMS No. Description Qty UM Net price Net worth VAT [%]'


In [100]:
for keyword in ["may", "jones", "diaz", "seller"]:
    position = text.lower().find(keyword)

    print(
        keyword,
        position,
        repr(
            text[
                max(0, position - 50):
                position + 100
            ]
        )
        if position >= 0
        else "not found",
    )

may 844 ',29 1 576,18 Total $ 1 432,89 $ 143,29 $ 1 576,18 May Way pro'
jones -1 not found
diaz -1 not found
seller 47 'Invoice no: 92944130 Date of issue: 07/07/2014 Seller: Client: Fritz , and Gomez Long, Jacobson and Roman 9654 Mary 204 Ward Harbors Jacksonhaven, '


In [102]:
from collections import defaultdict

entity_stats = defaultdict(
    lambda: {
        "expected": 0,
        "matched": 0,
        "ambiguous": 0,
    }
)

for _, row in annotated_df.iterrows():
    text = row["ocred_text"]
    target = json.loads(row["json_data"])

    for entity, value in extract_baseline_fields(target):
        entity_stats[entity]["expected"] += 1

        matches = find_value_matches(
            text,
            entity,
            value,
        )

        if len(matches) >= 1:
            entity_stats[entity]["matched"] += 1

        if len(matches) > 1:
            entity_stats[entity]["ambiguous"] += 1

entity_alignment_df = pd.DataFrame([
    {
        "entity": entity,
        "expected": values["expected"],
        "matched": values["matched"],
        "match_rate": (
            values["matched"]
            / values["expected"]
            if values["expected"]
            else 0.0
        ),
        "ambiguous": values["ambiguous"],
        "ambiguous_rate": (
            values["ambiguous"]
            / values["expected"]
            if values["expected"]
            else 0.0
        ),
    }
    for entity, values in entity_stats.items()
]).sort_values("match_rate")

entity_alignment_df.style.format({
    "match_rate": "{:.2%}",
    "ambiguous_rate": "{:.2%}",
})

,entity,expected,matched,match_rate,ambiguous,ambiguous_rate
3,SELLER_ADDRESS,1413,0,0.00%,0,0.00%
5,CLIENT_ADDRESS,1413,0,0.00%,0,0.00%
4,CLIENT_NAME,1413,1395,98.73%,0,0.00%
2,SELLER_NAME,1413,1400,99.08%,0,0.00%
0,INVOICE_NUMBER,1413,1413,100.00%,0,0.00%
1,INVOICE_DATE,1413,1413,100.00%,0,0.00%
6,TAX,1413,1413,100.00%,1405,99.43%
7,TOTAL,1413,1413,100.00%,1406,99.50%


In [112]:
RELIABLE_ENTITIES = set(
    entity_alignment_df.loc[
        entity_alignment_df["match_rate"] >= 0.90,
        "entity",
    ]
)

print(RELIABLE_ENTITIES)

{'INVOICE_DATE', 'SELLER_ADDRESS', 'TOTAL', 'INVOICE_NUMBER', 'TAX', 'CLIENT_ADDRESS', 'CLIENT_NAME', 'SELLER_NAME'}


### Inspect the distribution:

In [224]:
annotated_df["alignment_coverage"].value_counts(bins=10).sort_index()

print(
    "Coverage >= 0.9:",
    (annotated_df["alignment_coverage"] >= 0.9).mean(),
)

print(
    "Coverage == 1.0:",
    (annotated_df["alignment_coverage"] == 1.0).mean(),
)

Coverage >= 0.9: 0.9780608634111819
Coverage == 1.0: 0.9780608634111819


### For the initial baseline:

In [225]:
ner_df = (
    annotated_df.copy()
    .reset_index(drop=True)
)

print("Original documents:", len(annotated_df))
print("NER documents:", len(ner_df))
print("Removed:", len(annotated_df) - len(ner_df))

Original documents: 1413
NER documents: 1413
Removed: 0


### Inspect missing fields:

In [226]:
from collections import Counter

missing_counts = Counter(
    entity
    for entities in annotated_df["missing_entities"]
    for entity in entities
)

pd.Series(missing_counts).sort_values(ascending=False).rename("missing_count").to_frame()


,missing_count
CLIENT_NAME,18
SELLER_NAME,13


In [227]:
from collections import defaultdict

entity_stats = defaultdict(
    lambda: {
        "expected": 0,
        "matched": 0,
        "partially_matched": 0,
        "ambiguous": 0,
    }
)

for _, row in annotated_df.iterrows():
    text = row["ocred_text"]
    target = json.loads(row["json_data"])

    for entity, value in extract_baseline_fields(target):
        entity_stats[entity]["expected"] += 1

        parts = get_value_parts(entity, value)

        part_match_counts = [
            len(
                find_value_matches(
                    text,
                    entity,
                    part,
                )
            )
            for part in parts
        ]

        matched_parts = sum(
            count >= 1
            for count in part_match_counts
        )

        if matched_parts == len(parts):
            entity_stats[entity]["matched"] += 1

        elif matched_parts > 0:
            entity_stats[entity][
                "partially_matched"
            ] += 1

        if any(
            count > 1
            for count in part_match_counts
        ):
            entity_stats[entity]["ambiguous"] += 1

entity_alignment_df = pd.DataFrame([
    {
        "entity": entity,
        **values,
        "match_rate": (
            values["matched"]
            / values["expected"]
        ),
        "partial_rate": (
            values["partially_matched"]
            / values["expected"]
        ),
        "ambiguous_rate": (
            values["ambiguous"]
            / values["expected"]
        ),
    }
    for entity, values in entity_stats.items()
]).sort_values("match_rate")


entity_alignment_df.style.format({
    "match_rate": "{:.2%}",
    "partial_rate": "{:.2%}",
    "ambiguous_rate": "{:.2%}",
})


,entity,expected,matched,partially_matched,ambiguous,match_rate,partial_rate,ambiguous_rate
3,SELLER_ADDRESS,1413,1357,56,0,96.04%,3.96%,0.00%
5,CLIENT_ADDRESS,1413,1374,39,0,97.24%,2.76%,0.00%
4,CLIENT_NAME,1413,1395,0,0,98.73%,0.00%,0.00%
2,SELLER_NAME,1413,1400,0,0,99.08%,0.00%,0.00%
0,INVOICE_NUMBER,1413,1413,0,0,100.00%,0.00%,0.00%
1,INVOICE_DATE,1413,1413,0,0,100.00%,0.00%,0.00%
6,TAX,1413,1413,0,1405,100.00%,0.00%,99.43%
7,TOTAL,1413,1413,0,1406,100.00%,0.00%,99.50%


In [228]:
overlap_errors = []

for row_index, row in ner_df.iterrows():
    target = json.loads(row["json_data"])
    spans = find_entity_spans(
        row["ocred_text"],
        target,
    )

    for previous, current in zip(
        spans,
        spans[1:],
    ):
        if current["start"] < previous["end"]:
            overlap_errors.append({
                "row_index": row_index,
                "previous": previous,
                "current": current,
            })

print("Overlapping span errors:", len(overlap_errors))

Overlapping span errors: 0


In [229]:
sample = ner_df.sample(10, random_state=42)

for _, row in sample.iterrows():
    target = json.loads(row["json_data"])
    spans = find_entity_spans(
        row["ocred_text"],
        target,
    )

    financial_spans = [
        span
        for span in spans
        if span["entity"] in {"TAX", "TOTAL"}
    ]

    print(row["file_name"])

    for span in financial_spans:
        context = row["ocred_text"][
            max(0, span["start"] - 40):
            span["end"] + 40
        ]

        print(span["entity"], repr(context))

    print("=" * 80)

batch1-0897.jpg
TAX '0% 310,57 31,06 341,63 Total $ 310,57 $ 31,06 $ 341,63'
TOTAL '7 31,06 341,63 Total $ 310,57 $ 31,06 $ 341,63'
batch1-0896.jpg
TAX '0% 889,67 88,97 978,64 Total $ 889,67 $ 88,97 $ 978,64 Sexy'
TOTAL '7 88,97 978,64 Total $ 889,67 $ 88,97 $ 978,64 Sexy'
batch1-0412.jpg
TAX '0% 258,96 25,90 284,86 Total $ 258,96 $ 25,90 $ 284,86'
TOTAL '6 25,90 284,86 Total $ 258,96 $ 25,90 $ 284,86'
batch1-0028.jpg
TAX '6,58 103,66 1 140,24 Total $ 1 036,58 $ 103,66 $ 1 140,24 Sega'
TOTAL '66 1 140,24 Total $ 1 036,58 $ 103,66 $ 1 140,24 Sega'
batch1-0025.jpg
TAX '0,00 720,00 7 920,00 Total $ 7 200,00 $ 720,00 $ 7 920,00'
TOTAL '00 7 920,00 Total $ 7 200,00 $ 720,00 $ 7 920,00'
batch1-0851.jpg
TAX '9,97 475,00 5 224,97 Total $ 4 749,97 $ 475,00 $ 5 224,97'
TOTAL '00 5 224,97 Total $ 4 749,97 $ 475,00 $ 5 224,97'
batch1-0533.jpg
TAX ' 1 528,34 16 811,70 Total $ 15 283,36 $ 1 528,34 $ 16 811,70'
TOTAL '6 811,70 Total $ 15 283,36 $ 1 528,34 $ 16 811,70'
batch1-0260.jpg
TAX ' 930,85 93,

## Split data

In [231]:
train_df, temporary_df = train_test_split(
    ner_df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["subdir"],
)

validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temporary_df["subdir"],
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

Train: 1130
Validation: 141
Test: 142


## Tokenizer

In [115]:
MODEL_NAME = "google-bert/bert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

In [230]:
def money_digits(value):
    return "".join(
        re.findall(r"\d", str(value))
    )


financial_checks = []

for _, row in annotated_df.iterrows():
    expected = dict(
        extract_baseline_fields(row["target"])
    )

    spans = find_entity_spans(
        row["ocred_text"],
        row["target"],
    )

    predicted = {
        span["entity"]: span["value"]
        for span in spans
        if span["entity"] in {"TAX", "TOTAL"}
    }

    for entity in ["TAX", "TOTAL"]:
        expected_value = expected.get(entity)
        observed_value = predicted.get(entity)

        financial_checks.append({
            "file_name": row["file_name"],
            "entity": entity,
            "expected": expected_value,
            "observed": observed_value,
            "span_found": observed_value is not None,
            "value_matches_json": (
                observed_value is not None
                and money_digits(observed_value)
                == money_digits(expected_value)
            ),
        })


financial_checks_df = pd.DataFrame(
    financial_checks
)

financial_summary = (
    financial_checks_df
    .groupby("entity")
    .agg(
        documents=("file_name", "size"),
        spans_found=("span_found", "sum"),
        values_matching_json=("value_matches_json", "sum"),
    )
)

financial_summary["span_coverage"] = (
    financial_summary["spans_found"]
    / financial_summary["documents"]
)

financial_summary["ocr_value_accuracy"] = (
    financial_summary["values_matching_json"]
    / financial_summary["documents"]
)


financial_summary.style.format({
    "span_coverage": "{:.2%}",
    "ocr_value_accuracy": "{:.2%}",
})

,documents,spans_found,values_matching_json,span_coverage,ocr_value_accuracy
entity,,,,,
TAX,1413,1413,1409,100.00%,99.72%
TOTAL,1413,1413,1385,100.00%,98.02%


## Convert character spans into token labels

In [167]:
WINDOW_STRIDE = 64

def get_chunk_range(offsets):
    valid_offsets = [
        (start, end)
        for start, end in offsets
        if start != end
    ]

    if not valid_offsets:
        return None

    return (
        valid_offsets[0][0],
        valid_offsets[-1][1],
    )


def calculate_overlap(span, chunk_range):
    if chunk_range is None:
        return 0

    chunk_start, chunk_end = chunk_range

    return max(
        0,
        min(span["end"], chunk_end)
        - max(span["start"], chunk_start),
    )


def tokenize_and_label_batch(examples):
    output = {
        name: []
        for name in tokenizer.model_input_names
    }
    output["labels"] = []

    for text, json_data in zip(
        examples["ocred_text"],
        examples["json_data"],
    ):
        text = str(text)
        target = json.loads(json_data)

        spans = find_entity_spans(text, target)

        encoding = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            stride=WINDOW_STRIDE,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
        )

        offset_chunks = encoding.pop("offset_mapping")
        encoding.pop("overflow_to_sample_mapping", None)

        chunk_ranges = [
            get_chunk_range(offsets)
            for offsets in offset_chunks
        ]

        # Assign every entity to exactly one chunk. This prevents the
        # stride overlap from duplicating the entity in evaluation.
        span_owners = {}

        for span_index, span in enumerate(spans):
            fully_containing_chunks = [
                chunk_index
                for chunk_index, chunk_range
                in enumerate(chunk_ranges)
                if (
                    chunk_range is not None
                    and chunk_range[0] <= span["start"]
                    and span["end"] <= chunk_range[1]
                )
            ]

            if fully_containing_chunks:
                span_owners[span_index] = (
                    fully_containing_chunks[0]
                )
                continue

            overlaps = [
                calculate_overlap(span, chunk_range)
                for chunk_range in chunk_ranges
            ]

            best_chunk = int(np.argmax(overlaps))

            if overlaps[best_chunk] > 0:
                span_owners[span_index] = best_chunk

        for chunk_index, offsets in enumerate(offset_chunks):
            labels = []
            started_spans = set()

            for token_start, token_end in offsets:
                if token_start == token_end:
                    labels.append(-100)
                    continue

                overlapping_span = None

                for span_index, span in enumerate(spans):
                    overlaps = (
                        token_start < span["end"]
                        and token_end > span["start"]
                    )

                    if overlaps:
                        overlapping_span = (
                            span_index,
                            span,
                        )
                        break

                if overlapping_span is None:
                    labels.append(label2id["O"])
                    continue

                span_index, span = overlapping_span

                # This token belongs to a duplicate copy of the entity
                # caused by the overlapping window. Ignore it in loss.
                if span_owners.get(span_index) != chunk_index:
                    labels.append(-100)
                    continue

                prefix = (
                    "B"
                    if span_index not in started_spans
                    else "I"
                )

                labels.append(
                    label2id[
                        f"{prefix}-{span['entity']}"
                    ]
                )

                started_spans.add(span_index)

            for name in tokenizer.model_input_names:
                output[name].append(
                    encoding[name][chunk_index]
                )

            output["labels"].append(labels)

    return output

## Build Hugging Face datasets

In [232]:
def dataframe_to_dataset(frame):
    return Dataset.from_pandas(
        frame[["ocred_text", "json_data"]],
        preserve_index=False,
    )

raw_datasets = DatasetDict({
    "train": dataframe_to_dataset(train_df),
    "validation": dataframe_to_dataset(validation_df),
    "test": dataframe_to_dataset(test_df),
})

tokenized_datasets = raw_datasets.map(
    tokenize_and_label_batch,
    batched=True,
    batch_size=16,
    remove_columns=[
        "ocred_text",
        "json_data",
    ],
    load_from_cache_file=False,
)

tokenized_datasets

Map:   0%|          | 0/1130 [00:00<?, ? examples/s]

Map:   0%|          | 0/141 [00:00<?, ? examples/s]

Map:   0%|          | 0/142 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1130
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 141
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 142
    })
})

In [233]:
def count_entities(dataset):
    counts = Counter()

    for example in dataset:
        for label_id in example["labels"]:
            if label_id == -100:
                continue

            label = id2label[int(label_id)]

            if label.startswith("B-"):
                counts[label[2:]] += 1

    return pd.Series(counts).sort_values(ascending=False)


train_support = count_entities(
    tokenized_datasets["train"]
)

validation_support = count_entities(
    tokenized_datasets["validation"]
)

test_support = count_entities(
    tokenized_datasets["test"]
)

pd.DataFrame({
    "train": count_entities(
        tokenized_datasets["train"]
    ),
    "validation": count_entities(
        tokenized_datasets["validation"]
    ),
    "test": count_entities(
        tokenized_datasets["test"]
    ),
}).fillna(0).astype(int)

,train,validation,test
CLIENT_ADDRESS,2229,278,280
CLIENT_NAME,1113,140,142
INVOICE_DATE,1130,141,142
INVOICE_NUMBER,1130,141,142
SELLER_ADDRESS,2214,279,277
SELLER_NAME,1119,141,140
TAX,1130,141,142
TOTAL,1130,141,142


In [118]:
sample = tokenized_datasets["train"][0]

tokens = tokenizer.convert_ids_to_tokens(
    sample["input_ids"]
)

for token, label_id in zip(
    tokens,
    sample["labels"],
):
    if label_id != -100:
        print(
            f"{token:20s}",
            id2label[label_id],
        )

in                   O
##vo                 O
##ice                O
no                   O
:                    O
1758                 B-INVOICE_NUMBER
##54                 I-INVOICE_NUMBER
##15                 I-INVOICE_NUMBER
date                 O
of                   O
issue                O
:                    O
09                   B-INVOICE_DATE
/                    I-INVOICE_DATE
26                   I-INVOICE_DATE
/                    I-INVOICE_DATE
2016                 I-INVOICE_DATE
seller               O
:                    O
client               O
:                    O
eric                 B-SELLER_NAME
##kson               I-SELLER_NAME
,                    I-SELLER_NAME
hunt                 I-SELLER_NAME
and                  I-SELLER_NAME
gonzales             I-SELLER_NAME
carey                B-CLIENT_NAME
-                    I-CLIENT_NAME
montgomery           I-CLIENT_NAME
us                   B-SELLER_ADDRESS
##n                  I-SELLER_ADDRESS
##v             

## Initialize BERT

In [119]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

## Data collator

In [120]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)

## Evaluate metrics

In [121]:
def entity_metrics(true_sequences, predicted_sequences):
    true_positive = 0
    false_positive = 0
    false_negative = 0

    for true_labels, predicted_labels in zip(
        true_sequences,
        predicted_sequences,
    ):
        true_entities = bio_to_entities(true_labels)
        predicted_entities = bio_to_entities(
            predicted_labels
        )

        true_positive += len(
            true_entities & predicted_entities
        )

        false_positive += len(
            predicted_entities - true_entities
        )

        false_negative += len(
            true_entities - predicted_entities
        )

    precision = (
        true_positive
        / (true_positive + false_positive)
        if true_positive + false_positive
        else 0.0
    )

    recall = (
        true_positive
        / (true_positive + false_negative)
        if true_positive + false_negative
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    return precision, recall, f1

In [122]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    true_sequences = []
    predicted_sequences = []

    correct_tokens = 0
    evaluated_tokens = 0

    for prediction_row, label_row in zip(
        predictions,
        labels,
    ):
        true_labels = []
        predicted_labels = []

        for prediction_id, label_id in zip(
            prediction_row,
            label_row,
        ):
            if label_id == -100:
                continue

            true_label = id2label[int(label_id)]
            predicted_label = id2label[
                int(prediction_id)
            ]

            true_labels.append(true_label)
            predicted_labels.append(predicted_label)

            correct_tokens += (
                predicted_label == true_label
            )
            evaluated_tokens += 1

        true_sequences.append(true_labels)
        predicted_sequences.append(
            predicted_labels
        )

    precision, recall, f1 = entity_metrics(
        true_sequences,
        predicted_sequences,
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "token_accuracy": (
            correct_tokens / evaluated_tokens
            if evaluated_tokens
            else 0.0
        ),
    }

## Training configuration

In [124]:
import math
import transformers

print("Transformers:", transformers.__version__)

TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
NUM_EPOCHS = 8

micro_batches_per_epoch = math.ceil(
    len(train_df) / TRAIN_BATCH_SIZE
)

optimizer_steps_per_epoch = math.ceil(
    micro_batches_per_epoch
    / GRADIENT_ACCUMULATION_STEPS
)

total_training_steps = (
    optimizer_steps_per_epoch * NUM_EPOCHS
)

warmup_steps = int(total_training_steps * 0.10)

print("Optimizer steps per epoch:", optimizer_steps_per_epoch)
print("Total optimizer steps:", total_training_steps)
print("Warmup steps:", warmup_steps)

Transformers: 5.18.0
Optimizer steps per epoch: 71
Total optimizer steps: 568
Warmup steps: 56


In [125]:
OUTPUT_DIR = "/content/drive/MyDrive/ocr/bert"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

     learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=warmup_steps,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=(
        GRADIENT_ACCUMULATION_STEPS
    ),

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    save_total_limit=2,

    bf16=(
        torch.cuda.is_available()
        and torch.cuda.is_bf16_supported()
    ),
    fp16=(
        torch.cuda.is_available()
        and not torch.cuda.is_bf16_supported()
    ),

    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

## Train

In [126]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],

    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
        )
    ],
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Token Accuracy
1,0.571147,0.129122,0.320537,0.297683,0.308688,0.948826
2,0.056965,0.013072,0.965849,0.983066,0.974382,0.998208
3,0.035888,0.009736,0.992908,0.998217,0.995556,0.998771
4,0.023273,0.008937,0.992908,0.998217,0.995556,0.998771
5,0.022899,0.008744,0.987622,0.995544,0.991567,0.998694


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=355, training_loss=0.4433727585933578, metrics={'train_runtime': 68.8226, 'train_samples_per_second': 131.352, 'train_steps_per_second': 8.253, 'total_flos': 1099295908382616.0, 'train_loss': 0.4433727585933578, 'epoch': 5.0})

In [127]:
trainer.save_model(
    f"{OUTPUT_DIR}/best-model"
)

tokenizer.save_pretrained(
    f"{OUTPUT_DIR}/best-model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/ocr/bert/best-model/tokenizer_config.json',
 '/content/drive/MyDrive/ocr/bert/best-model/tokenizer.json')

## Test evaluation

In [128]:
test_metrics = trainer.evaluate(
    tokenized_datasets["test"],
    metric_key_prefix="test",
)

test_metrics

[transformers] early stopping required metric_for_best_model, but did not find eval_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Token Accuracy
0.022899,0.014806,5,0.985928,0.997331,0.991597,0.997679


{'test_loss': 0.01480635441839695,
 'test_precision': 0.9859278803869833,
 'test_recall': 0.9973309608540926,
 'test_f1': 0.9915966386554623,
 'test_token_accuracy': 0.9976787490753258}

## Extract entities from new text

In [244]:
def predict_entities(text):
    text = str(text)

    encoding = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        stride=WINDOW_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=True,
        return_tensors="pt",
    )

    offset_chunks = encoding.pop(
        "offset_mapping"
    ).tolist()

    encoding.pop(
        "overflow_to_sample_mapping",
        None,
    )

    model_inputs = {
        key: value.to(model.device)
        for key, value in encoding.items()
    }

    model.eval()

    with torch.no_grad():
        logits = (
            model(**model_inputs)
            .logits
            .cpu()
        )

    # Average predictions for tokens repeated in overlapping chunks.
    logits_by_offset = defaultdict(list)

    for chunk_logits, offsets in zip(
        logits,
        offset_chunks,
    ):
        for token_logits, (start, end) in zip(
            chunk_logits,
            offsets,
        ):
            if start == end:
                continue

            logits_by_offset[
                (start, end)
            ].append(token_logits)

    token_predictions = []

    for (start, end), repeated_logits in sorted(
        logits_by_offset.items()
    ):
        mean_logits = torch.stack(
            repeated_logits
        ).mean(dim=0)

        predicted_id = int(
            mean_logits.argmax()
        )

        token_predictions.append({
            "start": start,
            "end": end,
            "label": id2label[predicted_id],
        })

    entities = []
    current = None

    for token in token_predictions:
        label = token["label"]

        if label == "O":
            if current is not None:
                entities.append(current)
                current = None
            continue

        prefix, entity_type = label.split(
            "-",
            1,
        )

        starts_new = (
            prefix == "B"
            or current is None
            or current["type"] != entity_type
        )

        if starts_new:
            if current is not None:
                entities.append(current)

            current = {
                "type": entity_type,
                "start": token["start"],
                "end": token["end"],
            }
        else:
            current["end"] = token["end"]

    if current is not None:
        entities.append(current)

    for entity in entities:
        entity["text"] = text[
            entity["start"]:
            entity["end"]
        ]

    return entities

## Results

```json
{
    'test_loss': 0.01480635441839695,
    'test_precision': 0.9859278803869833,
    'test_recall': 0.9973309608540926,
    'test_f1': 0.9915966386554623,
    'test_token_accuracy': 0.9976787490753258
}
```

## Verify issue with saving

In [130]:

import torch
import transformers
import safetensors

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("safetensors:", safetensors.__version__)
print("best checkpoint:", trainer.state.best_model_checkpoint)

torch: 2.11.0+cu130
transformers: 5.18.0
safetensors: 0.8.0
best checkpoint: /content/drive/MyDrive/ocr/bert/checkpoint-213


In [131]:
from pathlib import Path
from safetensors import safe_open

best_checkpoint = Path(trainer.state.best_model_checkpoint)
model_file = best_checkpoint / "model.safetensors"

with safe_open(model_file, framework="pt", device="cpu") as f:
    layer_norm_keys = [
        key for key in f.keys()
        if "LayerNorm" in key
    ]

layer_norm_keys[:10]

['bert.embeddings.LayerNorm.beta',
 'bert.embeddings.LayerNorm.gamma',
 'bert.encoder.layer.0.attention.output.LayerNorm.beta',
 'bert.encoder.layer.0.attention.output.LayerNorm.gamma',
 'bert.encoder.layer.0.output.LayerNorm.beta',
 'bert.encoder.layer.0.output.LayerNorm.gamma',
 'bert.encoder.layer.1.attention.output.LayerNorm.beta',
 'bert.encoder.layer.1.attention.output.LayerNorm.gamma',
 'bert.encoder.layer.1.output.LayerNorm.beta',
 'bert.encoder.layer.1.output.LayerNorm.gamma']

In [132]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModelForTokenClassification

state_dict = load_file(model_file)

fixed_state_dict = {
    key.replace("LayerNorm.gamma", "LayerNorm.weight")
       .replace("LayerNorm.beta", "LayerNorm.bias"): value
    for key, value in state_dict.items()
}

config = AutoConfig.from_pretrained(best_checkpoint)
fixed_model = AutoModelForTokenClassification.from_config(config)

missing, unexpected = fixed_model.load_state_dict(
    fixed_state_dict,
    strict=False,
)

print("Missing:", missing)
print("Unexpected:", unexpected)

Missing: []
Unexpected: []


In [133]:
corrected_dir = Path(OUTPUT_DIR) / "best-model-corrected"

fixed_model.save_pretrained(
    corrected_dir,
    safe_serialization=True,
)
tokenizer.save_pretrained(corrected_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/ocr/bert/best-model-corrected/tokenizer_config.json',
 '/content/drive/MyDrive/ocr/bert/best-model-corrected/tokenizer.json')

In [134]:
verified_model = AutoModelForTokenClassification.from_pretrained(
    corrected_dir
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [135]:
verified_model

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

## Evaluation metrics

### Per-entity precision, recall, and F1

In [136]:

import numpy as np
import pandas as pd

from seqeval.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)


def decode_predictions(eval_prediction):
    logits, label_ids = eval_prediction
    predicted_ids = np.argmax(logits, axis=-1)

    predicted_labels = []
    actual_labels = []

    for predicted_sequence, actual_sequence in zip(
        predicted_ids,
        label_ids,
    ):
        sequence_predictions = []
        sequence_actual = []

        for predicted_id, actual_id in zip(
            predicted_sequence,
            actual_sequence,
        ):
            # -100 marks padding/special/subword tokens ignored by loss.
            if actual_id == -100:
                continue

            sequence_predictions.append(
                id2label[int(predicted_id)]
            )
            sequence_actual.append(
                id2label[int(actual_id)]
            )

        predicted_labels.append(sequence_predictions)
        actual_labels.append(sequence_actual)

    return predicted_labels, actual_labels


def compute_metrics(eval_prediction):
    predicted_labels, actual_labels = decode_predictions(
        eval_prediction
    )

    return {
        "precision": precision_score(
            actual_labels,
            predicted_labels,
            zero_division=0,
        ),
        "recall": recall_score(
            actual_labels,
            predicted_labels,
            zero_division=0,
        ),
        "f1": f1_score(
            actual_labels,
            predicted_labels,
            zero_division=0,
        ),
        "token_accuracy": accuracy_score(
            actual_labels,
            predicted_labels,
        ),
    }

In [137]:
test_output = trainer.predict(tokenized_datasets["test"])

predicted_labels, actual_labels = decode_predictions(
    (
        test_output.predictions,
        test_output.label_ids,
    )
)

report = classification_report(
    actual_labels,
    predicted_labels,
    output_dict=True,
    digits=4,
    zero_division=0,
)

report_df = pd.DataFrame(report).transpose()


report_df.style.format({
    "precision": "{:.2%}",
    "recall": "{:.2%}",
    "f1-score": "{:.2%}",
    "support": "{:.0f}",
})

,precision,recall,f1-score,support
CLIENT_ADDRESS,97.89%,99.64%,98.76%,280
CLIENT_NAME,99.30%,99.30%,99.30%,142
INVOICE_DATE,100.00%,100.00%,100.00%,142
INVOICE_NUMBER,100.00%,100.00%,100.00%,142
SELLER_ADDRESS,97.54%,100.00%,98.75%,277
SELLER_NAME,98.59%,100.00%,99.29%,140
TAX,0.00%,0.00%,0.00%,1
micro avg,98.59%,99.73%,99.16%,1124
macro avg,84.76%,85.56%,85.16%,1124
weighted avg,98.51%,99.73%,99.12%,1124


### &#32;Convert predicted entities into JSON

In [245]:
from collections import defaultdict

ENTITY_PATHS = {
    "INVOICE_NUMBER": ("invoice_number",),
    "INVOICE_DATE": ("invoice_date",),
    "SELLER_NAME": ("seller", "name"),
    "SELLER_ADDRESS": ("seller", "address"),
    "CLIENT_NAME": ("client", "name"),
    "CLIENT_ADDRESS": ("client", "address"),
    "TAX": ("tax",),
    "TOTAL": ("total",),
    "ACCOUNT_NUMBER": ("account_number",),
}

MULTILINE_ENTITIES = {
    "SELLER_ADDRESS",
    "CLIENT_ADDRESS",
}


def set_nested_value(document, path, value):
    current = document

    for key in path[:-1]:
        current = current.setdefault(key, {})

    current[path[-1]] = value


def entities_to_json(entities):
    """
    Supports your prediction format:

    {
        "type": "INVOICE_NUMBER",
        "start": 12,
        "end": 20,
        "text": "78566408"
    }
    """
    grouped = defaultdict(list)

    # Ensure document-order aggregation.
    entities = sorted(entities, key=lambda item: item["start"])

    for entity in entities:
        entity_type = (
            entity.get("type")
            or entity.get("entity")
            or entity.get("entity_group")
        )

        value = (
            entity.get("text")
            or entity.get("value")
            or entity.get("word")
        )

        if entity_type in ENTITY_PATHS and value:
            grouped[entity_type].append(value.strip())

    document = {}

    for entity_type, values in grouped.items():
        if entity_type in MULTILINE_ENTITIES:
            value = "\n".join(values)

        elif entity_type in {"TAX", "TOTAL"}:
            value = values[-1]

        else:
            value = values[0]

        set_nested_value(
            document,
            ENTITY_PATHS[entity_type],
            value,
        )

    return document

In [140]:
text = test_df.iloc[0]["ocred_text"]

predicted_entities = predict_entities(text)
predicted_json = entities_to_json(predicted_entities)

predicted_json

{'invoice_number': '78566408',
 'invoice_date': '03/18/2016',
 'seller': {'name': 'Clark PLC',
  'address': '3577 John Glen Suite 454\nNorth Michaelborough, NE 70613'},
 'client': {'name': 'Adams and Sons',
  'address': '957 Flowers Drive Apt: 985\nEast Jared, NJ 51167'}}

### Display actual and predicted JSON together

In [143]:
from IPython.display import display, JSON

In [142]:
def expected_pairs_to_json(expected_pairs):
    entities = []

    for index, (entity_type, value) in enumerate(expected_pairs):
        entities.append({
            "type": entity_type,
            "text": value,
            "start": index,
        })

    return entities_to_json(entities)

In [146]:
row = test_df.iloc[0]

# Replace this call with the function you currently use to produce
# the "Expected: [...]" output.
expected_pairs = extract_baseline_fields(row["target"])

actual_json = expected_pairs_to_json(expected_pairs)

predicted_entities = predict_entities(row["ocred_text"])
predicted_json = entities_to_json(predicted_entities)

print("Actual")
display(JSON(actual_json, expanded=True))

print("Predicted")
display(JSON(predicted_json, expanded=True))

Actual


<IPython.core.display.JSON object>

Predicted


<IPython.core.display.JSON object>

In [148]:
import json

comparison = pd.DataFrame({
    "actual": [
        json.dumps(actual_json, indent=2, ensure_ascii=False)
    ],
    "predicted": [
        json.dumps(predicted_json, indent=2, ensure_ascii=False)
    ],
})

comparison

,actual,predicted
0,"{\n ""invoice_number"": ""78566408"",\n ""invoice...","{\n ""invoice_number"": ""78566408"",\n ""invoice..."


### Normalized per-field exact match

BIO F1 can be high even when the reconstructed field is wrong. For example, an address can have almost every token correct but still fail exact matching.

In [149]:
import re
import unicodedata
from decimal import Decimal, InvalidOperation

MONEY_FIELDS = {"tax", "total", "subtotal"}
IDENTIFIER_FIELDS = {"invoice_number", "account_number"}


def normalize_money(value):
    value = unicodedata.normalize("NFKC", str(value))
    value = re.sub(r"[^\d,.\-]", "", value)

    if "," in value and "." in value:
        # The last separator is assumed to be the decimal separator.
        if value.rfind(",") > value.rfind("."):
            value = value.replace(".", "").replace(",", ".")
        else:
            value = value.replace(",", "")
    elif "," in value:
        value = value.replace(",", ".")

    try:
        return str(Decimal(value).normalize())
    except InvalidOperation:
        return value


def normalize_value(field, value):
    if value is None:
        return None

    value = unicodedata.normalize("NFKC", str(value)).strip()

    leaf_name = field.split(".")[-1]

    if leaf_name in MONEY_FIELDS:
        return normalize_money(value)

    if leaf_name in IDENTIFIER_FIELDS:
        return re.sub(r"[^A-Za-z0-9]", "", value).upper()

    # Normalize line breaks, repeated whitespace and case.
    value = re.sub(r"\s+", " ", value)
    value = value.casefold()

    # Treat common OCR punctuation variants consistently.
    value = value.replace(":", ".")
    value = re.sub(r"\s*([,.;])\s*", r"\1", value)

    return value


def flatten_json(document, prefix=""):
    result = {}

    for key, value in document.items():
        path = f"{prefix}.{key}" if prefix else key

        if isinstance(value, dict):
            result.update(flatten_json(value, path))
        elif not isinstance(value, list):
            result[path] = value

    return result

In [150]:
def evaluate_json_records(records):
    """
    records = [
        {
            "actual": {...},
            "predicted": {...},
        },
        ...
    ]
    """
    field_names = sorted({
        field
        for record in records
        for field in (
            set(flatten_json(record["actual"]))
            | set(flatten_json(record["predicted"]))
        )
    })

    rows = []

    for field in field_names:
        tp = fp = fn = 0
        expected_count = 0

        for record in records:
            actual = flatten_json(record["actual"]).get(field)
            predicted = flatten_json(record["predicted"]).get(field)

            actual_present = actual not in (None, "")
            predicted_present = predicted not in (None, "")

            if actual_present:
                expected_count += 1

            correct = (
                actual_present
                and predicted_present
                and normalize_value(field, actual)
                == normalize_value(field, predicted)
            )

            if correct:
                tp += 1
            else:
                if predicted_present:
                    fp += 1
                if actual_present:
                    fn += 1

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )

        rows.append({
            "field": field,
            "support": expected_count,
            "exact_matches": tp,
            "exact_match_rate": (
                tp / expected_count if expected_count else np.nan
            ),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

    return pd.DataFrame(rows)

In [152]:
def build_evaluation_records(test_df):
    records = []

    for row_index, row in test_df.iterrows():
        expected_pairs = extract_baseline_fields(row["target"])

        actual_json = expected_pairs_to_json(expected_pairs)
        predicted_json = entities_to_json(
            predict_entities(row["ocred_text"])
        )

        records.append({
            "row_index": row_index,
            "actual": actual_json,
            "predicted": predicted_json,
        })

    return records

In [153]:
evaluation_records = build_evaluation_records(test_df)
field_report = evaluate_json_records(evaluation_records)


field_report.style.format({
    "exact_match_rate": "{:.2%}",
    "precision": "{:.2%}",
    "recall": "{:.2%}",
    "f1": "{:.2%}",
})


,field,support,exact_matches,exact_match_rate,precision,recall,f1
0,client.address,142,134,94.37%,94.37%,94.37%,94.37%
1,client.name,142,140,98.59%,99.29%,98.59%,98.94%
2,invoice_date,142,142,100.00%,100.00%,100.00%,100.00%
3,invoice_number,142,142,100.00%,100.00%,100.00%,100.00%
4,seller.address,142,128,90.14%,90.14%,90.14%,90.14%
5,seller.name,142,139,97.89%,97.89%,97.89%,97.89%
6,tax,142,0,0.00%,0.00%,0.00%,0.00%
7,total,142,0,0.00%,0.00%,0.00%,0.00%


In [154]:
def document_is_exact(record):
    actual = flatten_json(record["actual"])
    predicted = flatten_json(record["predicted"])

    # Evaluate fields that are annotated in this document.
    for field, expected_value in actual.items():
        predicted_value = predicted.get(field)

        if (
            normalize_value(field, expected_value)
            != normalize_value(field, predicted_value)
        ):
            return False

    return True

In [156]:
document_results = [
    {
        "row_index": record["row_index"],
        "all_fields_correct": document_is_exact(record),
    }
    for record in evaluation_records
]

document_results = pd.DataFrame(document_results)
print(
    "Document exact accuracy:",
    document_results["all_fields_correct"].mean()
)

Document exact accuracy: 0.0


In [157]:
failed_indices = document_results.loc[
    ~document_results["all_fields_correct"],
    "row_index",
]

failed_records = [
    record
    for record in evaluation_records
    if record["row_index"] in set(failed_indices)
]

sample_failure = failed_records[0]

print("Actual")
display(JSON(sample_failure["actual"], expanded=True))

print("Predicted")
display(JSON(sample_failure["predicted"], expanded=True))

Actual


<IPython.core.display.JSON object>

Predicted


<IPython.core.display.JSON object>

In [158]:
def evaluate_text_source(frame, text_column):
    records = []

    for row_index, row in frame.iterrows():
        expected_pairs = extract_baseline_fields(row["target"])

        actual_json = expected_pairs_to_json(expected_pairs)
        predicted_json = entities_to_json(
            predict_entities(row[text_column])
        )

        records.append({
            "row_index": row_index,
            "actual": actual_json,
            "predicted": predicted_json,
        })

    report = evaluate_json_records(records)
    document_accuracy = np.mean([
        document_is_exact(record)
        for record in records
    ])

    return {
        "records": records,
        "field_report": report,
        "document_accuracy": document_accuracy,
    }

In [159]:
reference_result = evaluate_text_source(
    test_df,
    text_column="ocred_text",
)

paddle_result = evaluate_text_source(
    test_df,
    text_column="paddle_text",
)

pd.DataFrame([
    {
        "input": "reference_text",
        "document_accuracy": reference_result["document_accuracy"],
    },
    {
        "input": "paddleocr",
        "document_accuracy": paddle_result["document_accuracy"],
    },
])

KeyError: 'paddle_text'

## Add TAX and TOTAL fields

In [234]:
set_seed(SEED)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

In [235]:
import math

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
NUM_EPOCHS = 8

micro_batches_per_epoch = math.ceil(
    len(tokenized_datasets["train"])
    / TRAIN_BATCH_SIZE
)

optimizer_steps_per_epoch = math.ceil(
    micro_batches_per_epoch
    / GRADIENT_ACCUMULATION_STEPS
)

total_training_steps = (
    optimizer_steps_per_epoch
    * NUM_EPOCHS
)

warmup_steps = int(
    total_training_steps * 0.10
)

print(
    "Training chunks:",
    len(tokenized_datasets["train"]),
)
print(
    "Optimizer steps per epoch:",
    optimizer_steps_per_epoch,
)
print(
    "Maximum training steps:",
    total_training_steps,
)
print("Warmup steps:", warmup_steps)

Training chunks: 1130
Optimizer steps per epoch: 71
Maximum training steps: 568
Warmup steps: 56


In [236]:
OUTPUT_DIR = (
    "/content/drive/MyDrive/ocr/"
    "bert-tax-total-v2"
)


In [237]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=warmup_steps,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=(
        GRADIENT_ACCUMULATION_STEPS
    ),

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,

    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,

    save_total_limit=2,

    bf16=(
        torch.cuda.is_available()
        and torch.cuda.is_bf16_supported()
    ),
    fp16=(
        torch.cuda.is_available()
        and not torch.cuda.is_bf16_supported()
    ),

    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

In [238]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],

    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
        )
    ],
)

In [239]:
train_result = trainer.train()

print(train_result)
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint,
)
print(
    "Best validation metric:",
    trainer.state.best_metric,
)

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Token Accuracy
1,0.806508,0.171463,0.406003,0.366619,0.385307,0.936846
2,0.072813,0.013987,0.962885,0.980742,0.971731,0.997850
3,0.041364,0.008263,0.992913,0.999287,0.996090,0.998950
4,0.022662,0.007383,0.992913,0.999287,0.996090,0.998950
5,0.025027,0.006984,0.991501,0.998573,0.995025,0.998925


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=355, training_loss=0.4864709933039168, metrics={'train_runtime': 68.773, 'train_samples_per_second': 131.447, 'train_steps_per_second': 8.259, 'total_flos': 1099295908382616.0, 'train_loss': 0.4864709933039168, 'epoch': 5.0})
Best checkpoint: /content/drive/MyDrive/ocr/bert-tax-total-v2/checkpoint-213
Best validation metric: 0.9960895840739423


In [240]:
best_checkpoint = Path(
    trainer.state.best_model_checkpoint
)

state_dict = load_file(
    best_checkpoint / "model.safetensors"
)

fixed_state_dict = {
    key.replace(
        "LayerNorm.gamma",
        "LayerNorm.weight",
    ).replace(
        "LayerNorm.beta",
        "LayerNorm.bias",
    ): value
    for key, value in state_dict.items()
}

config = AutoConfig.from_pretrained(
    best_checkpoint
)

best_model = (
    AutoModelForTokenClassification
    .from_config(config)
)

missing, unexpected = best_model.load_state_dict(
    fixed_state_dict,
    strict=False,
)

print("Missing:", missing)
print("Unexpected:", unexpected)

Missing: []
Unexpected: []


In [241]:
corrected_dir = (
    Path(OUTPUT_DIR)
    / "best-model-corrected"
)

best_model.save_pretrained(
    corrected_dir,
    safe_serialization=True,
)

tokenizer.save_pretrained(
    corrected_dir
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = best_model.to(device)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [246]:
eval_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/evaluation",

    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    eval_strategy="no",
    save_strategy="no",

    bf16=(
        torch.cuda.is_available()
        and torch.cuda.is_bf16_supported()
    ),
    fp16=(
        torch.cuda.is_available()
        and not torch.cuda.is_bf16_supported()
    ),

    report_to="none",
    seed=SEED,
)

evaluation_trainer = Trainer(
    model=model,
    args=eval_args,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

test_output = evaluation_trainer.predict(
    tokenized_datasets["test"],
    metric_key_prefix="test",
)

In [247]:
test_output.metrics

{'test_loss': 0.014806770719587803,
 'test_model_preparation_time': 0.0042,
 'test_precision': 0.9873328641801549,
 'test_recall': 0.9971570717839374,
 'test_f1': 0.9922206506364922,
 'test_token_accuracy': 0.9976787490753258,
 'test_runtime': 0.6174,
 'test_samples_per_second': 229.999,
 'test_steps_per_second': 29.155}

In [248]:
predicted_labels, actual_labels = decode_predictions(
    (
        test_output.predictions,
        test_output.label_ids,
    )
)

report = classification_report(
    actual_labels,
    predicted_labels,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).transpose()


report_df.style.format({
    "precision": "{:.2%}",
    "recall": "{:.2%}",
    "f1-score": "{:.2%}",
    "support": "{:.0f}",
})

,precision,recall,f1-score,support
CLIENT_ADDRESS,97.54%,99.29%,98.41%,280
CLIENT_NAME,100.00%,99.30%,99.65%,142
INVOICE_DATE,100.00%,100.00%,100.00%,142
INVOICE_NUMBER,100.00%,100.00%,100.00%,142
SELLER_ADDRESS,97.19%,100.00%,98.58%,277
SELLER_NAME,98.59%,100.00%,99.29%,140
TAX,100.00%,100.00%,100.00%,142
TOTAL,99.30%,99.30%,99.30%,142
micro avg,98.73%,99.72%,99.22%,1407
macro avg,99.08%,99.73%,99.40%,1407


In [249]:
evaluation_records = build_evaluation_records(
    test_df
)

field_report = evaluate_json_records(
    evaluation_records
)


field_report.style.format({
    "exact_match_rate": "{:.2%}",
    "precision": "{:.2%}",
    "recall": "{:.2%}",
    "f1": "{:.2%}",
})


,field,support,exact_matches,exact_match_rate,precision,recall,f1
0,client.address,142,133,93.66%,93.66%,93.66%,93.66%
1,client.name,142,140,98.59%,99.29%,98.59%,98.94%
2,invoice_date,142,142,100.00%,100.00%,100.00%,100.00%
3,invoice_number,142,142,100.00%,100.00%,100.00%,100.00%
4,seller.address,142,127,89.44%,89.44%,89.44%,89.44%
5,seller.name,142,139,97.89%,97.89%,97.89%,97.89%
6,tax,142,142,100.00%,100.00%,100.00%,100.00%
7,total,142,140,98.59%,98.59%,98.59%,98.59%


In [250]:
field_report.loc[
    field_report["field"].isin(
        ["tax", "total"]
    )
].style.format({
    "exact_match_rate": "{:.2%}",
    "precision": "{:.2%}",
    "recall": "{:.2%}",
    "f1": "{:.2%}",
})

,field,support,exact_matches,exact_match_rate,precision,recall,f1
6,tax,142,142,100.00%,100.00%,100.00%,100.00%
7,total,142,140,98.59%,98.59%,98.59%,98.59%
